# DIMER Workshop: Comparing Tabular Foundation Models for Seven-Day Retail Demand Regression

**Dataset:** `freshretailnet-h7` sample dataset derived from FreshRetailNet-50K  
**Dataset source:** `examples/sample-data/freshretailnet-h7.zip` in `kurtvalcorza/mitra-regressor-pipeline`  
**Pinned dataset revision:** `78e12407044bbca6ed8edbb9754bb33bf09117ac`  
**Dataset acquisition:** Download the pinned repository artifact automatically and verify its SHA-256 before use  
**Model execution:** Run the regressors locally inside this notebook—no DIMER model job is required  
**Task:** Predict continuous `sale_amount` seven days ahead  
**Recommended audience:** College students with introductory Python and machine-learning experience  
**Estimated time:** Workshop duration must be measured in the intended runtime; allow extra time for first-time environment installation, depending on the number of foundation models selected and whether their environments are already cached

---

## What you will do

In this notebook, you will:

1. acquire and verify the fixed FreshRetailNet-derived sample artifact from the repository;
2. inspect and validate the data;
3. perform exploratory data analysis (EDA);
4. train several classical regression baselines;
5. install and run multiple tabular foundation-model regressors directly in Colab;
6. compare all models with the same evaluation protocol;
7. investigate whether stockout information affects prediction error; and
8. write an evidence-based conclusion that states what the experiment does—and does not—show.

> **Important:** This is a training and demonstration exercise. The sample is not an authoritative benchmark, and its target is **observed future sales**, not necessarily uncensored customer demand.


**Revision 2.1 — dataset-source and runtime remediation.** This version uses the pinned repository-hosted `freshretailnet-h7.zip` sample directly, verifies its content hash, and removes the DIMER-upload path. It also replaces the Python 3.13 foundation-model skip with isolated Python 3.12 environments bootstrapped through `uv`, and makes model-environment/runtime failures explicit instead of silently producing zero foundation-model runs.

The earlier identity-checked caches, retained artifacts, explicit freeze, no-refit final testing, and allowlisted report export remain in place.

**Validation scope:** static notebook checks and integration safeguards do not substitute for a clean end-to-end Colab execution with the real model checkpoints. Before teaching, execute the selected foundation-model conditions in a fresh runtime and confirm saved-artifact reloads.


## Research-design context

The activity uses a comparative experimental pattern:

- define one prediction task;
- hold the dataset and evaluation protocol constant;
- compare multiple model families;
- evaluate them with common metrics; and
- interpret whether additional model complexity provides meaningful value.

### Main research question

> How do tabular foundation models compare with conventional machine-learning baselines when predicting observed retail sales seven days ahead?

### Secondary questions

1. Does in-context use of an open-weight tabular foundation model outperform simple and classical baselines?
2. Does task-specific fine-tuning improve performance relative to an in-context condition?
3. Does prediction error change when stockout exposure is high?

## Learning objectives

By the end of this activity, you should be able to:

- distinguish training, validation, and test data;
- explain why chronological splitting matters for time-dependent data;
- identify target leakage and avoid it;
- interpret MAE, RMSE, median absolute error, and \(R^2\);
- compare models under a controlled protocol;
- distinguish **in-context learning** from **fine-tuning**;
- identify why a complex model must be compared with meaningful baselines;
- interpret model errors in relation to stockout conditions; and
- state conclusions with appropriate limitations.

## Short glossary

| Term | Meaning in this notebook |
|---|---|
| **Regression** | Predicting a continuous numerical value. |
| **Baseline** | A simple reference method used to determine whether a more complex model adds value. |
| **Tabular foundation model** | A pretrained model designed to learn patterns across tabular datasets and adapt to a new table through context, fine-tuning, or both. |
| **In-context learning (ICL)** | The model uses labelled examples as context without updating its pretrained weights. |
| **Fine-tuning** | The model updates some or all pretrained weights using the activity dataset. |
| **Temporal leakage** | Future information accidentally influences model training or model selection. |
| **Validation set** | Data used to compare candidate settings and make model-selection decisions. |
| **Test set** | Data reserved for one final evaluation after choices are fixed. |
| **Stockout** | A period when a product is unavailable; observed sales may then be lower than latent customer demand. |

## Dataset and experiment summary

The published archive contains three files:

| Partition | Rows | Purpose |
|---|---:|---|
| `train.csv` | 4,180 | Fit classical models or provide support/fine-tuning data to foundation models |
| `val.csv` | 1,600 | Compare candidate models and settings |
| `test.csv` | 1,600 | Perform one final evaluation after the configuration is fixed |

Each row contains 17 input features and one continuous target:

\[
\text{features at day } t \longrightarrow \text{observed sale\_amount at } t+7
\]

The source construction uses a purged chronological split with seven-row embargoes between partitions. The embargo prevents a feature row near a boundary from referring to a target date inside the following partition.

### Feature groups

- **Historical sales:** `lag_1`, `lag_7`, `lag_14`, rolling means, and rolling standard deviation
- **Stockout:** current and recent stockout hours
- **Context:** discount, holiday/activity flags, weather, day of week, and month

### Regressors executed in this notebook

| Model | Default notebook mode | Optional notebook mode | Weight licence note |
|---|---|---|---|
| Mitra Regressor | In-context learning | GPU fine-tuning | Apache-2.0 |
| TabDPT v1.2 Regressor | In-context learning | — | Apache-2.0 |
| TabPFN-3 Regressor | In-context learning | Public notebook does not carry its private fine-tuner | Non-commercial v3 weights |
| TabICLv2 Regressor | In-context learning | GPU fine-tuning | BSD-3-Clause |

The notebook creates a separate Python environment for each foundation-model repository. This avoids dependency conflicts while still running every selected regressor from this notebook.

> **Licence reminder:** A model's weight licence and the dataset licence both matter. FreshRetailNet-50K is CC BY 4.0. TabPFN-3 v3 has non-commercial weight terms and should be clearly labelled in reports and downstream artifacts.

## Before you begin

### Requirements

- A current Google Colab runtime
- Internet access from Colab to download the pinned sample dataset, install `uv`, clone the pinned pipeline repositories, and download their pinned open-weight snapshots
- Several gigabytes of free runtime storage if you run all four foundation models
- A GPU runtime for optional Mitra or TabICLv2 fine-tuning; the default in-context conditions can run without a DIMER model job

### Dataset acquisition

Section 1 automatically downloads this fixed repository artifact:

`kurtvalcorza/mitra-regressor-pipeline/examples/sample-data/freshretailnet-h7.zip`

The notebook pins the repository revision and expected SHA-256. If the download fails, the archive is not a valid ZIP, or the checksum differs, execution stops rather than substituting another dataset.

The sample is derived from FreshRetailNet-50K and is intended for smoke-testing, learning, and workshop demonstrations. It is not benchmark evidence.

### Model installation and runtime

Each model repository pins a different dependency stack. The notebook therefore creates isolated virtual environments under `/content/dimer_tabular_workshop_v2/environments/`.

Current Colab runtimes may use Python 3.13. The selected pipeline set is not uniformly compatible with Python 3.13, so Section 5.2 uses `uv` to create isolated **Python 3.12** environments for the foundation models. This keeps the notebook kernel unchanged while providing a compatible interpreter for all four pinned model pipelines.

The first installation can take several minutes per selected model. Later runs in the same Colab session reuse the environments and downloaded weights. You can select only two or three models for a shorter workshop.

### Test-set rule

During EDA and model selection, this notebook hides test-target summaries by default. Keep them hidden until:

1. the feature set is fixed;
2. the model conditions are fixed;
3. the primary metric is fixed; and
4. no more tuning decisions will be made.

This rule protects the independence of the final evaluation.


# 0. Set up the Colab environment

In [ ]:
# @title 0.1 Install the optional LightGBM dependency

INSTALL_LIGHTGBM = True # @param {type:"boolean"}
LIGHTGBM_VERSION = "4.6.0" # @param {type:"string"}

import importlib.util
import subprocess
import sys

if INSTALL_LIGHTGBM and importlib.util.find_spec("lightgbm") is None:
    print(f"Installing lightgbm=={LIGHTGBM_VERSION} ...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"lightgbm=={LIGHTGBM_VERSION}"]
    )

print("Environment preparation complete.")

In [ ]:
# @title 0.2 Import libraries and set experiment controls

from __future__ import annotations

import copy
import pickle
import uuid
import importlib
import hashlib
import json
import os
import platform
import shlex
import shutil
import subprocess
import sys
import time
import zipfile
from importlib import metadata
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    LIGHTGBM_AVAILABLE = False
    LIGHTGBM_IMPORT_ERROR = repr(exc)

RANDOM_SEED = 0 # @param {type:"integer"}
PRIMARY_METRIC = "mae" # @param ["mae", "rmse", "r2"]
INCLUDE_TEST_TARGETS_IN_EDA = False # @param {type:"boolean"}
TEST_EVALUATION_COMPLETED = False  # Updated only by the frozen-test cell.

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

def package_version(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not installed"

SOFTWARE_VERSIONS = {
    "python": platform.python_version(),
    "numpy": package_version("numpy"),
    "pandas": package_version("pandas"),
    "matplotlib": package_version("matplotlib"),
    "scikit-learn": package_version("scikit-learn"),
    "lightgbm": package_version("lightgbm"),
}

print("Experiment controls")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Primary metric: {PRIMARY_METRIC}")
print("  Test results are scored only after an explicit experiment freeze.")
print("\nSoftware versions")
display(pd.Series(SOFTWARE_VERSIONS, name="version").to_frame())

### One workspace per experiment
The notebook writes only to its own child directories. It never deletes the downloaded source archive or a participant-selected directory. Starting a new experiment creates a new folder and leaves the previous one intact.

The code below handles file safety and experiment records. Run it once; you do not need to edit the helper functions. Model files are retained locally so final testing can use the same fitted models. Serialized model files must come from this notebook in your own runtime—not from an untrusted upload.

In [ ]:
# @title 0.3 Create a protected experiment workspace
# @markdown Leave Start new experiment off during an exercise. Turn it on only to begin a new experiment;
# @markdown previous files are preserved. Turn it off again before continuing.
START_NEW_EXPERIMENT = False # @param {type:"boolean"}

CORE_SOURCE = '"""Notebook-owned experiment bookkeeping. No network or model imports.\n\nLocal artifact hashes protect against accidental substitution, not malicious code\nwith access to the same runtime. Never load someone else\'s serialized model.\n"""\nfrom __future__ import annotations\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport pickle\nimport re\nimport shutil\nimport stat\nimport uuid\nimport zipfile\nfrom pathlib import Path, PurePosixPath\nfrom typing import Any\nimport numpy as np\nimport pandas as pd\n\nCORE_VERSION = "2.1.0"\nOWNER = "dimer-freshretailnet-workshop-v2"\nRESULT_COLUMNS = ["model_key", "model", "family", "condition", "partition", "mae", "rmse",\n                  "median_absolute_error", "r2", "runtime_seconds", "fit_runtime_seconds",\n                  "prediction_runtime_seconds", "effective_mode", "device", "licence", "notes", "run_id"]\n\n\ndef canonical_json(value: Any) -> str:\n    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False)\n\n\ndef fingerprint(value: Any) -> str:\n    return hashlib.sha256(canonical_json(value).encode()).hexdigest()\n\n\ndef sha256_file(path: str | Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write_json(path: str | Path, value: Any) -> None:\n    path = Path(path)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\\n", encoding="utf-8")\n    tmp.replace(path)\n\n\ndef checked_name(value: str) -> str:\n    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,100}", value) or value in {".", ".."}:\n        raise ValueError("Use a simple name (letters, numbers, dash, underscore); not a path.")\n    return value\n\n\ndef owned_root(path: str | Path) -> Path:\n    """Claim only an empty directory; never delete or take ownership of user data."""\n    path = Path(path).absolute()\n    if any(p.is_symlink() for p in (path, *path.parents)):\n        raise ValueError("The workshop workspace must not traverse symbolic links.")\n    if path == Path(path.anchor) or path == Path.home():\n        raise ValueError("Choose a dedicated workshop directory, not a root or home directory.")\n    marker = path / ".workshop-owner.json"\n    if path.exists():\n        if not path.is_dir():\n            raise ValueError("Workspace is not a directory.")\n        if marker.exists():\n            if json.loads(marker.read_text()).get("owner") != OWNER:\n                raise ValueError("This workspace belongs to another application.")\n        elif any(path.iterdir()):\n            raise ValueError("Refusing to claim a non-empty, unowned directory. No files were deleted.")\n    path.mkdir(parents=True, exist_ok=True)\n    if not marker.exists():\n        write_json(marker, {"owner": OWNER})\n    return path.resolve()\n\n\ndef new_owned_directory(parent: Path, label: str) -> Path:\n    """Create a fresh child. No operation in this module recursively deletes files."""\n    parent = Path(parent).resolve()\n    parent.mkdir(parents=True, exist_ok=True)\n    name = checked_name(label) + "-" + uuid.uuid4().hex[:12]\n    path = parent / name\n    path.mkdir(exist_ok=False)\n    write_json(path / ".workshop-owner.json", {"owner": OWNER})\n    return path\n\n\ndef stage_dataset_zip(archive_path: Path, parent: Path, *, max_bytes: int = 512 * 1024**2) -> dict[str, Path]:\n    """Accept flat or nested splits; stage exactly three CSVs under a fresh owned child."""\n    archive_path = Path(archive_path).resolve(strict=True)\n    if not zipfile.is_zipfile(archive_path):\n        raise ValueError("Upload the unchanged dataset ZIP, not an extracted CSV.")\n    required = {"train.csv", "val.csv", "test.csv"}\n    with zipfile.ZipFile(archive_path) as zf:\n        chosen = {}\n        expanded = 0\n        seen = set()\n        for info in zf.infolist():\n            name = info.filename\n            p = PurePosixPath(name)\n            if "\\\\" in name or p.is_absolute() or ".." in p.parts or name in seen:\n                raise ValueError(f"Unsafe or duplicate ZIP path: {name}")\n            if stat.S_ISLNK((info.external_attr >> 16) & 0xFFFF):\n                raise ValueError(f"ZIP symbolic links are not accepted: {name}")\n            if info.flag_bits & 1:\n                raise ValueError("Encrypted ZIP members are not accepted.")\n            seen.add(name)\n            expanded += info.file_size\n            if expanded > max_bytes:\n                raise ValueError("Archive exceeds the workshop\'s expanded-size limit.")\n            if not info.is_dir() and p.name in required:\n                if p.name in chosen:\n                    raise ValueError(f"Multiple copies of {p.name}; upload one unambiguous dataset.")\n                chosen[p.name] = info\n        if set(chosen) != required:\n            raise ValueError(f"Required ZIP members missing: {sorted(required - set(chosen))}")\n        # All paths and size checks complete before writing; CRC checked as each member is read.\n        destination = new_owned_directory(parent, "dataset")\n        paths = {}\n        for name, info in chosen.items():\n            payload = zf.read(info)\n            header = next(csv.reader(io.StringIO(payload.decode("utf-8-sig"))), [])\n            if not header or len(set(header)) != len(header):\n                raise ValueError(f"Empty/duplicate CSV headers in {name}.")\n            target = destination / name\n            target.write_bytes(payload)\n            paths[name[:-4]] = target.resolve()\n    return paths\n\n\ndef split_identity(paths: dict[str, Path], features: list[str], archive_sha: str) -> dict[str, Any]:\n    return {"archive_sha256": archive_sha, "features": list(features),\n            "split_sha256": {k: sha256_file(paths[k]) for k in ("train", "val", "test")}}\n\n\ndef regression_metrics(y_true: Any, y_pred: Any) -> dict[str, float | None]:\n    y = np.asarray(y_true, dtype=float).reshape(-1)\n    pred = np.asarray(y_pred, dtype=float).reshape(-1)\n    if y.shape != pred.shape or not y.size or not np.isfinite(y).all() or not np.isfinite(pred).all():\n        raise ValueError("Metrics require nonempty, equally sized, finite targets and predictions.")\n    error = y - pred\n    total = float(np.sum((y - y.mean())**2))\n    return {"mae": float(np.abs(error).mean()), "rmse": float(np.sqrt((error**2).mean())),\n            "median_absolute_error": float(np.median(np.abs(error))),\n            "r2": float(1 - (error**2).sum()/total) if total > 0 else None}\n\n\ndef prediction_frame(frame: pd.DataFrame, predictions: Any) -> pd.DataFrame:\n    values = np.asarray(predictions, dtype=float).reshape(-1)\n    if len(values) != len(frame) or not np.isfinite(values).all():\n        raise ValueError("Predictions must contain one finite number per input row.")\n    out = pd.DataFrame({"row_id": np.arange(len(frame)), "target": frame.target.to_numpy(float),\n                        "prediction": values})\n    for col in ("stockout_hours", "roll_7_stockout"):\n        if col in frame:\n            out[col] = frame[col].to_numpy()\n    return out\n\n\ndef canonical_predictions(frame: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:\n    """One public prediction schema; align by local split row_id, never by guessed order."""\n    out = frame.copy()\n    for old, new in (("observed", "target"), ("predicted", "prediction")):\n        if old in out and new not in out:\n            out = out.rename(columns={old: new})\n    if not {"row_id", "target", "prediction"}.issubset(out):\n        raise ValueError("Prediction CSV requires row_id, target, prediction.")\n    if len(out) != len(reference):\n        raise ValueError("Prediction row count differs from the reference split.")\n    ids = pd.to_numeric(out.row_id, errors="raise").to_numpy(float)\n    if not np.isfinite(ids).all() or not np.equal(ids, np.floor(ids)).all():\n        raise ValueError("row_id must contain finite integers.")\n    if sorted(ids.astype(int).tolist()) != list(range(len(reference))):\n        raise ValueError("Missing, duplicate, or out-of-range prediction row_id.")\n    out = out.sort_values("row_id").reset_index(drop=True)\n    for col in ("target", "prediction"):\n        out[col] = pd.to_numeric(out[col], errors="raise")\n        if not np.isfinite(out[col].to_numpy(float)).all():\n            raise ValueError(f"Non-finite {col} in prediction CSV.")\n    if not np.allclose(out.target, reference.target.to_numpy(float), rtol=1e-12, atol=1e-12):\n        raise ValueError("Prediction targets do not match the declared split rows.")\n    # Always use reference features for stratified diagnostics.\n    for col in ("stockout_hours", "roll_7_stockout"):\n        if col in reference:\n            out[col] = reference[col].to_numpy()\n    return out\n\n\ndef normalize_results(records: list[dict] | pd.DataFrame) -> pd.DataFrame:\n    result = pd.DataFrame(records).copy()\n    for col in RESULT_COLUMNS:\n        if col not in result:\n            result[col] = "cpu" if col == "device" else (None if col != "notes" else "")\n    for col in ("mae", "rmse", "median_absolute_error", "r2", "runtime_seconds"):\n        result[col] = pd.to_numeric(result[col], errors="coerce")\n    return result\n\n\ndef choose_diagnostic_model(registry: dict, requested: str = "") -> str | None:\n    if not registry:\n        return None\n    if requested and requested not in registry:\n        print(f"\'{requested}\' has no current predictions. Using the first available model instead.")\n    return requested if requested in registry else sorted(registry)[0]\n\n\ndef shift_statistics(reference: pd.Series, comparison: pd.Series) -> dict[str, Any]:\n    a = pd.to_numeric(reference, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()\n    b = pd.to_numeric(comparison, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()\n    result = {"reference_n": len(a), "comparison_n": len(b),\n              "reference_mean": float(a.mean()) if len(a) else None,\n              "comparison_mean": float(b.mean()) if len(b) else None,\n              "smd": None, "status": "insufficient_data"}\n    if len(a) < 2 or len(b) < 2:\n        return result\n    pooled = (a.var(ddof=1) + b.var(ddof=1))/2\n    delta = float(b.mean() - a.mean())\n    if pooled == 0:\n        result.update(smd=0.0 if delta == 0 else None,\n                      status="identical_constants" if delta == 0 else "constant_value_changed")\n    elif math.isfinite(pooled):\n        result.update(smd=float(delta/np.sqrt(pooled)), status="defined")\n    else:\n        result["status"] = "undefined_variance"\n    return result\n\n\ndef inventory(root: Path, *, exclude: tuple[str, ...] = ()) -> dict[str, str]:\n    result = {}\n    for path in sorted(Path(root).rglob("*")):\n        if path.is_symlink():\n            raise ValueError("Symbolic links are not permitted in experiment artifacts.")\n        if path.is_file():\n            rel = path.relative_to(root).as_posix()\n            if rel not in exclude:\n                result[rel] = sha256_file(path)\n    return result\n\n\ndef verify_inventory(root: Path, expected: dict[str, str]) -> None:\n    for rel, digest in expected.items():\n        p = PurePosixPath(rel)\n        if p.is_absolute() or ".." in p.parts or "\\\\" in rel:\n            raise ValueError("Unsafe artifact inventory path.")\n        file = root / rel\n        if file.is_symlink() or not file.is_file() or sha256_file(file) != digest:\n            raise ValueError(f"Missing or changed artifact: {rel}")\n\n\ndef validate_receipt(path: Path, *, expected_fingerprint: str | None = None,\n                     expected_receipt_sha: str | None = None) -> dict:\n    path = Path(path)\n    if expected_receipt_sha and sha256_file(path) != expected_receipt_sha:\n        raise ValueError("Run receipt changed after registration.")\n    receipt = json.loads(path.read_text())\n    if receipt.get("status") != "complete" or fingerprint(receipt["identity"]) != receipt["fingerprint"]:\n        raise ValueError("Run receipt is incomplete or its identity is invalid.")\n    if expected_fingerprint and receipt["fingerprint"] != expected_fingerprint:\n        raise ValueError("Cached result belongs to a different experiment/configuration.")\n    verify_inventory(path.parent, receipt["files"])\n    return receipt\n\n\nclass Session:\n    """One explicit experiment. Development writes stop after freeze."""\n    def __init__(self, root: Path, base_identity: dict):\n        self.root = Path(root).resolve()\n        self.state_path = self.root / "experiment.json"\n        if self.state_path.exists():\n            state = json.loads(self.state_path.read_text())\n            if state["base_identity"] != base_identity:\n                raise ValueError("Dataset, seed, metric, or features changed. Start a new experiment in 0.3.")\n        else:\n            state = {"experiment_id": uuid.uuid4().hex, "base_identity": base_identity,\n                     "records": {}, "freeze_sha256": None}\n            write_json(self.state_path, state)\n        self.experiment_id = state["experiment_id"]\n        self.base_identity = base_identity\n\n    def state(self) -> dict:\n        return json.loads(self.state_path.read_text())\n\n    def assert_development(self) -> None:\n        if (self.root / "freeze.json").exists():\n            raise RuntimeError("This experiment is frozen. Start a new experiment before fitting or tuning again.")\n\n    def find_cache(self, key: str, identity: dict) -> Path | None:\n        self.assert_development()\n        record = self.state()["records"].get("development:" + key)\n        if not record or record["fingerprint"] != fingerprint(identity):\n            return None\n        path = Path(record["receipt"])\n        validate_receipt(path, expected_fingerprint=fingerprint(identity),\n                         expected_receipt_sha=record["receipt_sha256"])\n        return path\n\n    def register(self, run_dir: Path, identity: dict, *, stage: str = "development") -> Path:\n        if stage == "development":\n            self.assert_development()\n        if identity.get("experiment_id") != self.experiment_id:\n            raise ValueError("Run belongs to another experiment.")\n        if stage == "test" and not self.state().get("freeze_sha256"):\n            raise ValueError("Cannot register test results before freezing.")\n        run_dir = Path(run_dir).resolve()\n        if self.root not in run_dir.parents:\n            raise ValueError("Run must be inside this experiment\'s directory.")\n        for filename in ("run_config.json", "result.json"):\n            if not (run_dir / filename).is_file():\n                raise ValueError(f"Incomplete run: missing {filename}")\n        receipt = {"status": "complete", "stage": stage, "identity": identity,\n                   "fingerprint": fingerprint(identity), "run_id": run_dir.name,\n                   "files": inventory(run_dir, exclude=("receipt.json", "run.log"))}\n        path = run_dir / "receipt.json"\n        write_json(path, receipt)\n        validate_receipt(path)\n        state = self.state()\n        state["records"][stage + ":" + identity["model_key"]] = {\n            "receipt": str(path), "receipt_sha256": sha256_file(path),\n            "fingerprint": receipt["fingerprint"]}\n        write_json(self.state_path, state)\n        return path\n\n    def current_record(self, key: str, stage: str = "development") -> tuple[Path, dict]:\n        record = self.state()["records"].get(stage + ":" + key)\n        if not record:\n            raise ValueError(f"No successful current {stage} run for {key}.")\n        path = Path(record["receipt"])\n        return path, validate_receipt(path, expected_receipt_sha=record["receipt_sha256"])\n\n    def freeze(self, keys: list[str], controls: dict) -> Path:\n        if not keys or len(keys) != len(set(keys)):\n            raise ValueError("Select one or more unique successful model keys.")\n        freeze_path = self.root / "freeze.json"\n        if freeze_path.exists():\n            previous = self.validate_freeze(controls)\n            if sorted(previous["selected"]) != sorted(keys):\n                raise ValueError("Selection changed after freeze. Start a new experiment.")\n            return freeze_path\n        selected = {}\n        for key in keys:\n            path, receipt = self.current_record(key)\n            if receipt["identity"].get("controls") != controls.get(key):\n                raise ValueError(f"{key}: settings changed since validation; rerun development before freezing.")\n            if not any(name.startswith("artifact/") for name in receipt["files"]):\n                raise ValueError(f"{key}: no retained fitted artifact.")\n            selected[key] = {"receipt": str(path), "receipt_sha256": sha256_file(path),\n                             "fingerprint": receipt["fingerprint"], "run_id": receipt["run_id"]}\n        document = {"experiment_id": self.experiment_id, "base_identity": self.base_identity,\n                    "selected": selected, "controls": {k: controls[k] for k in sorted(keys)},\n                    "protocol": "load_saved_artifact_no_refit"}\n        write_json(freeze_path, document)\n        state = self.state()\n        state["freeze_sha256"] = sha256_file(freeze_path)\n        write_json(self.state_path, state)\n        return freeze_path\n\n    def validate_freeze(self, controls: dict) -> dict:\n        path = self.root / "freeze.json"\n        expected = self.state().get("freeze_sha256")\n        if not expected or not path.exists() or sha256_file(path) != expected:\n            raise ValueError("No valid frozen experiment. Run the Freeze experiment cell first.")\n        document = json.loads(path.read_text())\n        for key, entry in document["selected"].items():\n            if controls.get(key) != document["controls"][key]:\n                raise ValueError(f"{key}: configuration changed after freeze. No test run was started.")\n            validate_receipt(Path(entry["receipt"]), expected_receipt_sha=entry["receipt_sha256"],\n                             expected_fingerprint=entry["fingerprint"])\n        return document\n\n    def export_records(self, keys: list[str], controls: dict, *, include_test: bool) -> list[Path]:\n        """Explicit allowlist only; never walks the shared cache."""\n        freeze = self.validate_freeze(controls) if include_test else None\n        output = []\n        for key in keys:\n            path, receipt = self.current_record(key)\n            if receipt["identity"].get("controls") != controls.get(key):\n                raise ValueError(f"Refusing to export stale development result: {key}")\n            output.append(path)\n            if include_test and key in freeze["selected"]:\n                test_path, test = self.current_record(key, "test")\n                expected = freeze["selected"][key]["receipt_sha256"]\n                if test["identity"].get("source_receipt_sha256") != expected:\n                    raise ValueError("Test result is not bound to the selected development artifact.")\n                if test["identity"].get("freeze_sha256") != self.state()["freeze_sha256"]:\n                    raise ValueError("Test result belongs to another freeze.")\n                output.append(test_path)\n        return output\n\n\ndef export_run_files(receipts: list[Path], destination: Path) -> None:\n    for path in receipts:\n        receipt = validate_receipt(path)\n        target = destination / "runs" / receipt["stage"] / receipt["run_id"]\n        target.mkdir(parents=True, exist_ok=False)\n        # Serialized models stay local. The receipt retains their hashes without redistributing weights.\n        for name in ("receipt.json", "run_config.json", "result.json", "validation_predictions.csv", "test_predictions.csv"):\n            if name != "receipt.json" and name not in receipt["files"]:\n                continue\n            file = path.parent / name\n            if file.is_file():\n                shutil.copy2(file, target / name)\n\n\ndef fit_mitra_with_chronological_validation(train: pd.DataFrame, validation: pd.DataFrame, *,\n                                          path: Path, fine_tune: bool, steps: int, time_limit: int,\n                                          seed: int, device: str, predictor_factory=None):\n    """Notebook-owned adapter overlay: explicit outer validation, no random AutoGluon holdout.\n\n    The pinned repository remains unchanged. Its verified snapshot loader is reused.\n    This explicit fit path is identified separately in run/adapter provenance.\n    """\n    if predictor_factory is None:\n        from autogluon.tabular import TabularPredictor\n        predictor_factory = TabularPredictor\n    if fine_tune and (device != "cuda" or steps <= 0):\n        raise ValueError("Mitra fine-tuning needs CUDA and a positive number of steps.")\n    hp = {"fine_tune": bool(fine_tune), "seed": int(seed), "metric": "mae", "device": device}\n    if fine_tune:\n        hp["fine_tune_steps"] = int(steps)\n    predictor = predictor_factory(label="target", problem_type="regression", eval_metric="mean_absolute_error",\n                                  path=str(path), verbosity=2)\n    predictor.fit(train_data=train, tuning_data=validation, hyperparameters={"MITRA": hp},\n                  num_bag_folds=0, num_stack_levels=0, fit_weighted_ensemble=False,\n                  dynamic_stacking=False, refit_full=False, time_limit=int(time_limit),\n                  num_gpus=1 if device == "cuda" else 0,\n                  ag_args_fit={"max_memory_usage_ratio": 1.10})\n    if not any("mitra" in n.lower() for n in predictor.model_names()):\n        raise RuntimeError("AutoGluon did not produce a Mitra model.")\n    train_x, train_y = predictor.load_data_internal(data="train")\n    val_x, val_y = predictor.load_data_internal(data="val")\n    counts = {"train_rows_supplied": len(train), "validation_rows_supplied": len(validation),\n              "autogluon_train_rows_observed": len(train_x), "autogluon_validation_rows_observed": len(val_x),\n              "internal_validation_policy": "explicit_val_csv_no_random_outer_holdout",\n              "effective_model_context_rows": None,\n              "context_note": "Observed counts are AutoGluon-level; per-query model context is not assumed identical."}\n    if len(train_x) != len(train) or len(val_x) != len(validation):\n        raise RuntimeError("Unexpected AutoGluon row counts; refusing an undocumented holdout or augmentation.")\n    return predictor, counts\n\n\nclass ReferenceRegressor:\n    """Pickle-safe naive baseline; predict never learns from the query targets."""\n    def __init__(self, value: float, feature: str | None = None):\n        self.value, self.feature = float(value), feature\n    def predict(self, frame: pd.DataFrame) -> np.ndarray:\n        if self.feature:\n            return pd.to_numeric(frame[self.feature], errors="coerce").fillna(self.value).to_numpy(float)\n        return np.full(len(frame), self.value)\n'
EXECUTION_SOURCE = '"""Identity-checked local execution. Imported by the notebook, not a hosted service."""\nfrom __future__ import annotations\nimport copy\nimport json\nimport os\nimport pickle\nimport subprocess\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom workshop_core import (Session, fingerprint, sha256_file, write_json, new_owned_directory,\n                           validate_receipt, canonical_predictions, prediction_frame,\n                           regression_metrics, normalize_results)\n\n\nclass LocalExperiment:\n    def __init__(self, session: Session, split_paths: dict, runner: Path, core: Path,\n                 *, resolve_environment, execute_command):\n        self.session=session\n        self.paths={k:Path(v).resolve() for k,v in split_paths.items()}\n        self.runner=Path(runner)\n        self.core=Path(core)\n        self.resolve_environment=resolve_environment\n        self.execute_command=execute_command\n\n    def identity(self, key: str, controls: dict, features: list[str]) -> dict:\n        return {"experiment_id":self.session.experiment_id,"model_key":key,\n                "split_sha256":{k:sha256_file(p) for k,p in sorted(self.paths.items())},\n                "archive_sha256":self.session.base_identity["dataset"]["archive_sha256"],\n                "features":list(features),"controls":copy.deepcopy(controls)}\n\n    def run_validation(self, key: str, controls: dict, spec: dict, *, force=False) -> Path:\n        self.session.assert_development()\n        identity=self.identity(key,controls,controls["features"])\n        cached=None if force else self.session.find_cache(key,identity)\n        if cached:\n            print(f"Reusing verified identical run: {key}")\n            return cached\n        environment=self.resolve_environment(spec["environment"])\n        if environment["signature"] != controls["environment_signature"]:\n            raise ValueError("Environment changed since configuration. Rerun environment preparation.")\n        folder=new_owned_directory(self.session.root / "runs",key)\n        config={"phase":"development","model_key":key,"display_name":spec["display_name"],\n                "condition":spec["condition"],"model_family":spec["model_family"],\n                "model_configuration":copy.deepcopy(controls["parameters"]),\n                "repository":controls["repository"],"repository_commit":controls["repository_commit"],\n                "repository_path":str(environment["repository_path"]),\n                "split_paths":{k:str(v) for k,v in self.paths.items()},\n                "output_dir":str(folder),"seed":controls["seed"],\n                "device_preference":controls["device_preference"],\n                "identity":identity,"identity_fingerprint":fingerprint(identity)}\n        write_json(folder / "run_config.json",config)\n        self.execute_command([str(environment["python"]),str(self.runner),"--config",str(folder / "run_config.json")],\n                             log_path=folder / "run.log")\n        result=json.loads((folder / "result.json").read_text())\n        if result.get("identity_fingerprint") != fingerprint(identity):\n            raise ValueError("Runner returned a result with the wrong experiment identity.")\n        reference=pd.read_csv(self.paths["val"],float_precision="round_trip")\n        predicted=canonical_predictions(pd.read_csv(folder / "validation_predictions.csv"),reference)\n        expected=regression_metrics(predicted.target,predicted.prediction)\n        for metric,value in expected.items():\n            if value is not None and not np.isclose(result["partition_metrics"]["validation"][metric],value):\n                raise ValueError("Runner metrics differ from predictions on the declared validation rows.")\n        if not result.get("reload_validation_check",{}).get("passed"):\n            raise ValueError("Fitted artifact did not pass its reload check.")\n        return self.session.register(folder,identity)\n\n    def run_test(self, key: str, controls: dict) -> Path:\n        freeze=self.session.validate_freeze(controls)\n        if key not in freeze["selected"]:\n            raise ValueError("Requested test model was not selected at freeze.")\n        for split,path in self.paths.items():\n            if sha256_file(path) != freeze["base_identity"]["dataset"]["split_sha256"][split]:\n                raise ValueError("Dataset changed after freeze; no test run started.")\n        selected=freeze["selected"][key]\n        original=Path(selected["receipt"])\n        receipt=validate_receipt(original,expected_receipt_sha=selected["receipt_sha256"])\n        if receipt["identity"]["controls"]["backend"] == "classical":\n            raise ValueError("Use the classical artifact evaluator for this model.")\n        source_config=json.loads((original.parent / "run_config.json").read_text())\n        environment=self.resolve_environment(controls[key]["environment"])\n        if environment["signature"] != controls[key]["environment_signature"]:\n            raise ValueError("Frozen environment changed; no model was refitted.")\n        identity=copy.deepcopy(receipt["identity"])\n        identity.update(freeze_sha256=self.session.state()["freeze_sha256"],\n                        source_receipt_sha256=selected["receipt_sha256"],phase="test")\n        existing=self.session.state()["records"].get("test:"+key)\n        if existing:\n            path=Path(existing["receipt"])\n            validate_receipt(path,expected_fingerprint=fingerprint(identity),expected_receipt_sha=existing["receipt_sha256"])\n            print(f"Using completed frozen test result: {key}")\n            return path\n        folder=new_owned_directory(self.session.root / "tests",key)\n        config=copy.deepcopy(source_config)\n        config.update(phase="test", output_dir=str(folder),source_receipt=str(original),\n                      source_receipt_sha256=selected["receipt_sha256"],source_fingerprint=receipt["fingerprint"],\n                      freeze_sha256=self.session.state()["freeze_sha256"],identity=identity,\n                      identity_fingerprint=fingerprint(identity))\n        write_json(folder / "run_config.json",config)\n        self.execute_command([str(environment["python"]),str(self.runner),"--config",str(folder / "run_config.json")],\n                             log_path=folder / "run.log")\n        result=json.loads((folder / "result.json").read_text())\n        if result.get("identity_fingerprint") != fingerprint(identity) or result.get("evaluation_protocol") != "loaded_selected_artifact_no_refit":\n            raise ValueError("Final result does not prove the declared frozen evaluation path.")\n        truth=pd.read_csv(self.paths["test"],float_precision="round_trip")\n        canonical_predictions(pd.read_csv(folder / "test_predictions.csv"),truth)\n        return self.session.register(folder,identity,stage="test")\n\n\ndef save_classical_run(session: Session, key: str, model, train: pd.DataFrame,\n                       validation: pd.DataFrame, controls: dict, *, fit_seconds: float,\n                       identity: dict, model_name: str, family="Classical ML", mode="supervised_fit") -> Path:\n    """Save the exact estimator that made the validation predictions; not a new fit."""\n    session.assert_development()\n    folder=new_owned_directory(session.root / "runs",key)\n    (folder / "artifact").mkdir()\n    with (folder / "artifact" / "fitted.pkl").open("wb") as f: pickle.dump(model,f,pickle.HIGHEST_PROTOCOL)\n    features=controls["features"]\n    start=time.perf_counter();pred=model.predict(validation[features]);elapsed=time.perf_counter()-start\n    # Local-only pickle. This freshly written object is trusted; later loads verify its registered hash.\n    with (folder / "artifact" / "fitted.pkl").open("rb") as f: recovered=pickle.load(f)\n    np.testing.assert_allclose(pred,recovered.predict(validation[features]),rtol=1e-12,atol=1e-12)\n    prediction_frame(validation,pred).to_csv(folder / "validation_predictions.csv",index=False)\n    result={"model_key":key,"model":model_name,"family":family,"condition":mode,\n            "effective_mode":mode,"device":"cpu","licence":"See library licences",\n            "identity_fingerprint":fingerprint(identity),"training_audit":{"train_rows_supplied":len(train)},\n            "runtime_seconds":{"fit":fit_seconds,"validation_prediction":elapsed},\n            "partition_metrics":{"validation":regression_metrics(validation.target,pred),"test":None},\n            "reload_validation_check":{"passed":True}}\n    write_json(folder / "result.json",result)\n    write_json(folder / "run_config.json",{"identity":identity,"controls":controls})\n    return session.register(folder,identity)\n\n\ndef evaluate_classical_frozen(session: Session, key: str, controls: dict, validation: pd.DataFrame,\n                              test: pd.DataFrame) -> Path:\n    freeze=session.validate_freeze(controls)\n    selected=freeze["selected"][key]\n    path=Path(selected["receipt"])\n    original=validate_receipt(path,expected_receipt_sha=selected["receipt_sha256"])\n    if original["identity"]["controls"]["backend"] != "classical":\n        raise ValueError("Not a classical model artifact.")\n    identity=copy.deepcopy(original["identity"])\n    identity.update(freeze_sha256=session.state()["freeze_sha256"],source_receipt_sha256=selected["receipt_sha256"],phase="test")\n    existing=session.state()["records"].get("test:"+key)\n    if existing:\n        p=Path(existing["receipt"])\n        validate_receipt(p,expected_fingerprint=fingerprint(identity),expected_receipt_sha=existing["receipt_sha256"])\n        return p\n    features=original["identity"]["features"]\n    # Hashes in the freeze were just checked before deserialization.\n    with (path.parent / "artifact" / "fitted.pkl").open("rb") as f: model=pickle.load(f)\n    old=canonical_predictions(pd.read_csv(path.parent / "validation_predictions.csv"),validation)\n    np.testing.assert_allclose(old.prediction,model.predict(validation[features]),rtol=1e-10,atol=1e-10)\n    start=time.perf_counter();pred=model.predict(test[features]);elapsed=time.perf_counter()-start\n    folder=new_owned_directory(session.root / "tests",key)\n    result=json.loads((path.parent / "result.json").read_text())\n    result.update(partition_metrics={"validation":None,"test":regression_metrics(test.target,pred)},\n                  runtime_seconds={"fit":0.0,"test_prediction":elapsed},identity_fingerprint=fingerprint(identity),\n                  source_receipt_sha256=selected["receipt_sha256"],freeze_sha256=session.state()["freeze_sha256"],\n                  evaluation_protocol="loaded_selected_artifact_no_refit")\n    prediction_frame(test,pred).to_csv(folder / "test_predictions.csv",index=False)\n    write_json(folder / "result.json",result)\n    write_json(folder / "run_config.json",{"identity":identity,"source_receipt":str(path)})\n    return session.register(folder,identity,stage="test")\n\n\ndef load_results(receipts: list[Path], reference: pd.DataFrame, partition="validation"):\n    records=[]; registry={}\n    for path in receipts:\n        path=Path(path); receipt=validate_receipt(path)\n        expected_stage="development" if partition=="validation" else "test"\n        if receipt["stage"] != expected_stage:\n            raise ValueError("Wrong partition/stage in results loader.")\n        result=json.loads((path.parent / "result.json").read_text())\n        frame=canonical_predictions(pd.read_csv(path.parent / f"{partition}_predictions.csv"),reference)\n        times=result.get("runtime_seconds",{})\n        metrics=regression_metrics(frame.target,frame.prediction)\n        records.append({"model_key":result["model_key"],"model":result["model"],"family":result["family"],\n                        "condition":result["condition"],"partition":partition,**metrics,\n                        "runtime_seconds":times.get("fit",0)+times.get(partition+"_prediction",0),\n                        "fit_runtime_seconds":times.get("fit",0),\n                        "prediction_runtime_seconds":times.get(partition+"_prediction",0),\n                        "effective_mode":result["effective_mode"],"device":result["device"],\n                        "licence":result.get("model_licence",result.get("licence")),"run_id":receipt["run_id"],\n                        "model_revision":result.get("model_revision"),"notes":"Current registered run"})\n        registry[result["model_key"]]=frame\n    return normalize_results(records),registry\n'

# These helper modules are carried inside this notebook; no extra download is required.
try:
    Path("/content").mkdir(parents=True, exist_ok=True)
    WORKSPACE_ROOT = Path("/content/dimer_tabular_workshop_v2")
except Exception:
    if Path("/kaggle/working").exists():
        WORKSPACE_ROOT = Path("/kaggle/working/dimer_tabular_workshop_v2")
    else:
        WORKSPACE_ROOT = (Path.cwd() / "dimer_tabular_workshop_v2").resolve()
# Load the bundled ownership helper before claiming any nonempty directory.
bootstrap={"__name__":"_workshop_bootstrap"}
exec(compile(CORE_SOURCE,"<bundled-workshop-core>","exec"),bootstrap)
WORKSPACE_ROOT=bootstrap["owned_root"](WORKSPACE_ROOT)
SUPPORT_REVISION=hashlib.sha256((CORE_SOURCE+EXECUTION_SOURCE).encode()).hexdigest()
SUPPORT_DIR=WORKSPACE_ROOT / "support" / SUPPORT_REVISION
SUPPORT_DIR.mkdir(parents=True,exist_ok=True)
for name,source in (("workshop_core",CORE_SOURCE),("workshop_execution",EXECUTION_SOURCE)):
    file=SUPPORT_DIR / (name+".py")
    if name in sys.modules and Path(sys.modules[name].__file__).resolve()!=file.resolve():
        raise RuntimeError("A different helper revision is already imported. Restart the runtime before switching notebook revisions.")
    file.write_bytes(source.encode("utf-8"))
if str(SUPPORT_DIR) not in sys.path: sys.path.insert(0,str(SUPPORT_DIR))
from workshop_core import (owned_root, new_owned_directory, stage_dataset_zip, sha256_file, write_json,
    fingerprint, split_identity, canonical_predictions, prediction_frame, normalize_results,
    choose_diagnostic_model, shift_statistics, Session, validate_receipt, export_run_files, ReferenceRegressor)
from workshop_execution import (LocalExperiment, save_classical_run, evaluate_classical_frozen, load_results)
if START_NEW_EXPERIMENT or "SESSION_ROOT" not in globals():
    SESSION_ROOT=new_owned_directory(WORKSPACE_ROOT / "experiments","experiment")
    FIGURE_REGISTRY={}
    TEST_EVALUATION_COMPLETED=False
    TEST_TARGETS_VIEWED=False
    CLASSICAL_RECEIPTS=[]
    FOUNDATION_RECEIPTS=[]
    ABLATION_RECEIPTS=[]
    FINAL_RECEIPTS=[]
    CURRENT_CLASSICAL_KEYS=[]
    CURRENT_FOUNDATION_KEYS=[]
    CURRENT_ABLATION_KEYS=[]
    print("Created a new experiment directory. Previous experiments were not deleted.")
print("Current experiment:",SESSION_ROOT)

def show_and_save_plot(run_ids=None):
    """Save the current figure before displaying it; export uses this registry, not a folder sweep."""
    figure=plt.gcf()
    title=figure.axes[0].get_title() if figure.axes else "figure"
    axis_key=title+"|"+(figure.axes[0].get_xlabel() if figure.axes else "")+"|"+(figure.axes[0].get_ylabel() if figure.axes else "")
    key=hashlib.sha256(axis_key.encode()).hexdigest()[:16]
    folder=SESSION_ROOT / "figures";folder.mkdir(exist_ok=True)
    path=folder / (key+"-"+uuid.uuid4().hex[:8]+".png")
    figure.savefig(path,dpi=150,bbox_inches="tight")
    FIGURE_REGISTRY[key]={"path":str(path),"sha256":sha256_file(path),"title":title,"run_ids":list(run_ids or [])}
    plt.show()
    plt.close(figure)

### Check your setup

The preceding cell should display the random seed, primary metric, and package versions. Record these values because software versions can affect model behavior and reproducibility.

# 1. Acquire the pinned sample dataset


This notebook uses the repository-hosted `freshretailnet-h7.zip` fixture directly. No DIMER dataset download or manual upload is required.

The canonical workshop artifact is:

- repository: `kurtvalcorza/mitra-regressor-pipeline`;
- path: `examples/sample-data/freshretailnet-h7.zip`;
- pinned revision: `78e12407044bbca6ed8edbb9754bb33bf09117ac`; and
- expected SHA-256: `6534230e9eb6a2e212b741c4c17d897a57338323eb011f35ba2dc3fb28d8bb7b`.

The archive is a small, leakage-aware sample derived from FreshRetailNet-50K for temporal panel regression. The notebook verifies the exact archive bytes before extraction, then checks that `train.csv`, `val.csv`, and `test.csv` are present and validates their expected row counts and schema in Section 2.

Because the URL is pinned to a repository commit rather than `main`, every participant should receive the same dataset bytes unless the download is intercepted or unavailable. A checksum mismatch causes the notebook to stop.


In [ ]:
# @title 1.1 Download and verify the pinned FreshRetailNet sample
DATASET_REPOSITORY = "kurtvalcorza/mitra-regressor-pipeline"
DATASET_REPOSITORY_PATH = "examples/sample-data/freshretailnet-h7.zip"
DATASET_REPOSITORY_REVISION = "78e12407044bbca6ed8edbb9754bb33bf09117ac"
DATASET_ARCHIVE_URL = (
    "https://raw.githubusercontent.com/"
    f"{DATASET_REPOSITORY}/{DATASET_REPOSITORY_REVISION}/"
    f"{DATASET_REPOSITORY_PATH}"
)
EXPECTED_SHA256 = "6534230e9eb6a2e212b741c4c17d897a57338323eb011f35ba2dc3fb28d8bb7b"

download_dir = new_owned_directory(SESSION_ROOT / "downloads", "freshretailnet")
archive_path = download_dir / "freshretailnet-h7.zip"

print(f"Downloading pinned dataset artifact from:\n{DATASET_ARCHIVE_URL}")
try:
    import urllib.request
    urllib.request.urlretrieve(DATASET_ARCHIVE_URL, str(archive_path))
except Exception as err:
    raise RuntimeError(
        "Could not download the pinned workshop dataset. "
        "Check the runtime's internet access and retry."
    ) from err

if not archive_path.is_file() or not zipfile.is_zipfile(archive_path):
    raise ValueError(f"Downloaded artifact is not a valid ZIP archive: {archive_path}")

DATASET_SHA256 = sha256_file(archive_path)
if DATASET_SHA256.lower() != EXPECTED_SHA256.lower():
    raise ValueError(
        "Dataset checksum mismatch. Refusing to continue with unexpected bytes. "
        f"Expected {EXPECTED_SHA256}, observed {DATASET_SHA256}."
    )

DATASET_PROVENANCE = {
    "dataset_name": "freshretailnet-h7 sample derived from FreshRetailNet-50K",
    "repository": DATASET_REPOSITORY,
    "repository_path": DATASET_REPOSITORY_PATH,
    "repository_revision": DATASET_REPOSITORY_REVISION,
    "acquisition_method": "Pinned raw GitHub repository artifact",
    "source_url": DATASET_ARCHIVE_URL,
    "archive_bytes": archive_path.stat().st_size,
    "sha256": DATASET_SHA256,
    "sha256_verified": True,
}
display(pd.Series(DATASET_PROVENANCE, name="value").to_frame())


### Expected result

You should see:

- the pinned repository revision;
- the exact repository path and source URL;
- the local archive path;
- the archive's byte size;
- the verified SHA-256 digest; and
- later, confirmation that `train.csv`, `val.csv`, and `test.csv` are present.

The SHA-256 check protects the workshop from silently using a changed or substituted sample artifact. The later schema and row-count checks verify that the extracted tables match the expected workshop contract.


In [ ]:
# @title 1.2 Validate the ZIP and stage the three CSV files
# @markdown Both flat and nested ZIP layouts are supported. Unrelated ZIP members are not extracted.
dataset_paths=stage_dataset_zip(archive_path,SESSION_ROOT / "inputs")
extract_directory=dataset_paths["train"].parent
print("Staged dataset:",extract_directory)
print("The original archive and all unrelated files were left unchanged.")
display(pd.Series({k:str(v) for k,v in dataset_paths.items()},name="CSV path").to_frame())

# 2. Load and validate the dataset

Validation is not the same as model evaluation. Here, validation means checking whether the files conform to the expected dataset contract:

- the required files exist;
- the three partitions share the same columns;
- the target is finite and numeric;
- the expected feature columns are present; and
- the row counts match the published data card.

A warning does not automatically mean the dataset is unusable. It means you should investigate before continuing.

In [ ]:
# @title 2.1 Load train, validation, and test CSV files

TARGET_COLUMN = "target"

EXPECTED_ROWS = {
    "train": 4180,
    "val": 1600,
    "test": 1600,
}

EXPECTED_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "roll_7_mean",
    "roll_28_mean",
    "roll_7_std",
    "stockout_hours",
    "roll_7_stockout",
    "discount",
    "holiday_flag",
    "activity_flag",
    "precpt",
    "avg_temperature",
    "avg_humidity",
    "avg_wind_level",
    "dow",
    "month",
]

frames = {
    split: pd.read_csv(path,float_precision="round_trip")
    for split, path in dataset_paths.items()
}

reference_columns = list(frames["train"].columns)

for split, frame in list(frames.items()):
    if set(frame.columns) != set(reference_columns):
        missing = sorted(set(reference_columns) - set(frame.columns))
        extra = sorted(set(frame.columns) - set(reference_columns))
        raise ValueError(
            f"{split} schema differs from train.csv. "
            f"Missing={missing}; extra={extra}"
        )

    frames[split] = frame[reference_columns]

if TARGET_COLUMN not in reference_columns:
    raise ValueError(f"Target column not found: {TARGET_COLUMN}")

feature_columns = [
    column for column in reference_columns
    if column != TARGET_COLUMN
]

missing_features = sorted(set(EXPECTED_FEATURES) - set(feature_columns))
unexpected_features = sorted(set(feature_columns) - set(EXPECTED_FEATURES))

if missing_features:
    raise ValueError(f"Expected features are missing: {missing_features}")

if unexpected_features:
    print(f"Warning: additional features found: {unexpected_features}")

for split, frame in frames.items():
    numeric_target = pd.to_numeric(frame[TARGET_COLUMN], errors="coerce")

    if numeric_target.isna().any():
        bad_count = int(numeric_target.isna().sum())
        raise ValueError(
            f"{split} contains {bad_count} missing or non-numeric target values."
        )

    if not np.isfinite(numeric_target.to_numpy()).all():
        raise ValueError(f"{split} contains infinite target values.")

    frames[split][TARGET_COLUMN] = numeric_target

    expected_rows = EXPECTED_ROWS[split]
    if len(frame) != expected_rows:
        print(
            f"Warning: {split} has {len(frame):,} rows; "
            f"the data card states {expected_rows:,}."
        )

X_train = frames["train"][feature_columns].copy()
y_train = frames["train"][TARGET_COLUMN].copy()

X_val = frames["val"][feature_columns].copy()
y_val = frames["val"][TARGET_COLUMN].copy()

X_test = frames["test"][feature_columns].copy()
y_test_sealed = frames["test"][TARGET_COLUMN].copy()

print("Dataset loaded successfully.")
print(f"Features: {len(feature_columns)}")
for split, frame in frames.items():
    print(f"{split:>5}: {len(frame):,} rows × {frame.shape[1]} columns")

print("\nTraining sample:")
display(frames["train"].head())
# Features must remain numeric for this fixed workshop artifact. Do not silently convert invalid strings.
for split,frame in frames.items():
    for feature in feature_columns:
        frame[feature]=pd.to_numeric(frame[feature],errors="raise")
        if np.isinf(frame[feature].to_numpy(float)).any(): raise ValueError(f"Infinite feature: {split}/{feature}")
    if len(frame)!=EXPECTED_ROWS[split]:
        raise ValueError(f"Wrong workshop sample: {split} has {len(frame)} rows; expected {EXPECTED_ROWS[split]}.")
DATASET_IDENTITY=split_identity(dataset_paths,feature_columns,DATASET_SHA256)
BASE_IDENTITY={"dataset":DATASET_IDENTITY,"seed":RANDOM_SEED,"primary_metric":PRIMARY_METRIC,
               "support_revision":SUPPORT_REVISION}
SESSION=Session(SESSION_ROOT,BASE_IDENTITY)

def assert_current_data():
    current={"dataset":split_identity(dataset_paths,feature_columns,sha256_file(archive_path)),
             "seed":RANDOM_SEED,"primary_metric":PRIMARY_METRIC,"support_revision":SUPPORT_REVISION}
    if current!=SESSION.base_identity:
        raise ValueError("Inputs or experiment controls changed. Start a new experiment in Section 0.3.")
    for split,frame in frames.items():
        original=pd.read_csv(dataset_paths[split],float_precision="round_trip")
        pd.testing.assert_frame_equal(frame.reset_index(drop=True),original[frame.columns].reset_index(drop=True),
                                      check_dtype=False,check_exact=False,rtol=1e-12,atol=1e-12)
    for name,split in (("X_train","train"),("X_val","val"),("X_test","test")):
        pd.testing.assert_frame_equal(globals()[name],frames[split][feature_columns],check_dtype=False)
    for name,split in (("y_train","train"),("y_val","val"),("y_test_sealed","test")):
        pd.testing.assert_series_equal(globals()[name],frames[split]["target"],check_dtype=False)
    if sha256_file(SUPPORT_DIR / "workshop_core.py") != hashlib.sha256(CORE_SOURCE.encode()).hexdigest():
        raise ValueError("Bundled core source changed on disk; start a fresh runtime.")
    if sha256_file(SUPPORT_DIR / "workshop_execution.py") != hashlib.sha256(EXECUTION_SOURCE.encode()).hexdigest():
        raise ValueError("Bundled execution source changed on disk; start a fresh runtime.")
    return current

def base_controls(features=None):
    assert_current_data()
    return {"dataset":DATASET_IDENTITY,"seed":RANDOM_SEED,"primary_metric":PRIMARY_METRIC,
            "features":list(feature_columns if features is None else features),
            "support_revision":SUPPORT_REVISION}


In [ ]:
# @title 2.2 Create a dataset-quality report

quality_rows = []

for split, frame in frames.items():
    quality_rows.append(
        {
            "split": split,
            "rows": len(frame),
            "columns": frame.shape[1],
            "feature_count": len(feature_columns),
            "missing_cells": int(frame.isna().sum().sum()),
            "duplicate_rows": int(frame.duplicated().sum()),
            "numeric_columns": int(
                frame.select_dtypes(include=np.number).shape[1]
            ),
        }
    )

quality_report = pd.DataFrame(quality_rows).set_index("split")
display(quality_report)

column_report = pd.DataFrame(
    {
        "dtype": frames["train"].dtypes.astype(str),
        "train_missing": frames["train"].isna().sum(),
        "val_missing": frames["val"].isna().sum(),
        "test_missing": frames["test"].isna().sum(),
        "train_unique": frames["train"].nunique(dropna=False),
    }
)

display(column_report)

## Why the supplied split must be retained

The dataset comes from time-dependent store–product observations. A random split can place later observations in training and earlier observations in validation or testing. That setup does not represent prospective forecasting.

The supplied split is:

```text
train | 7-row embargo | validation | 7-row embargo | test
```

Because the target is seven days in the future, the embargo removes rows whose target dates could cross a partition boundary.

> **Rule for this activity:** Do not concatenate and randomly resplit the three files.

# 3. Perform exploratory data analysis

EDA helps you understand the dataset before model fitting. It can reveal:

- skewed targets;
- outliers;
- missing values;
- redundant features;
- differences between time periods; and
- relationships that motivate later experiments.

EDA does not establish causality. A correlation between stockout hours and future sales, for example, may reflect demand, availability, replenishment policy, or several interacting mechanisms.

In [ ]:
# @title 3.1 Summarize the target without opening the test set

target_splits = ["train", "val"]

if INCLUDE_TEST_TARGETS_IN_EDA:
    TEST_TARGETS_VIEWED=True
    target_splits.append("test")

target_summary_rows = []

for split in target_splits:
    target = frames[split][TARGET_COLUMN]

    target_summary_rows.append(
        {
            "split": split,
            "count": len(target),
            "minimum": target.min(),
            "p01": target.quantile(0.01),
            "p25": target.quantile(0.25),
            "median": target.median(),
            "mean": target.mean(),
            "p75": target.quantile(0.75),
            "p99": target.quantile(0.99),
            "maximum": target.max(),
            "standard_deviation": target.std(),
            "zero_count": int(target.eq(0).sum()),
            "zero_rate_percent": 100 * target.eq(0).mean(),
            "negative_count": int(target.lt(0).sum()),
        }
    )

target_summary = pd.DataFrame(target_summary_rows).set_index("split")
display(target_summary.round(4))

### How to interpret the target summary

- **Mean versus median:** A much larger mean suggests a right-skewed distribution.
- **P99 versus maximum:** A large gap suggests extreme values.
- **Zero rate:** Zero-valued targets affect percentage-based metrics such as MAPE.
- **Negative count:** The data card describes zero and positive targets, so negative values would require investigation.

In [ ]:
# @title 3.2 Plot the target distribution

TARGET_VIEW = "log1p" # @param ["raw", "log1p"]
HISTOGRAM_BINS = 40 # @param {type:"slider", min:10, max:100, step:5}

plt.figure(figsize=(9, 5))

for split in target_splits:
    values = frames[split][TARGET_COLUMN].dropna()

    if TARGET_VIEW == "log1p":
        if values.lt(0).any():
            raise ValueError(
                "The log1p view cannot be used because negative targets are present."
            )
        values = np.log1p(values)
        x_label = "log1p(target)"
    else:
        x_label = TARGET_COLUMN

    plt.hist(
        values,
        bins=HISTOGRAM_BINS,
        density=True,
        histtype="step",
        linewidth=2,
        label=split,
    )

plt.title("Target distribution by partition")
plt.xlabel(x_label)
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
show_and_save_plot()

In [ ]:
# @title 3.3 Compare target spread and zero-valued observations

SHOW_BOXPLOT_OUTLIERS = False # @param {type:"boolean"}

plt.figure(figsize=(8, 5))
try:
    plt.boxplot(
        [frames[split][TARGET_COLUMN].dropna() for split in target_splits],
        tick_labels=target_splits,
        showfliers=SHOW_BOXPLOT_OUTLIERS,
    )
except TypeError:
    plt.boxplot(
        [frames[split][TARGET_COLUMN].dropna() for split in target_splits],
        labels=target_splits,
        showfliers=SHOW_BOXPLOT_OUTLIERS,
    )
plt.title("Target distribution by partition")
plt.xlabel("Partition")
plt.ylabel(TARGET_COLUMN)
plt.tight_layout()
show_and_save_plot()

zero_rates = [
    100 * frames[split][TARGET_COLUMN].eq(0).mean()
    for split in target_splits
]

plt.figure(figsize=(7, 5))
plt.bar(target_splits, zero_rates)
plt.title("Zero-valued targets by partition")
plt.xlabel("Partition")
plt.ylabel("Zero-valued targets (%)")
plt.tight_layout()
show_and_save_plot()

## Feature distribution shift

The validation and test partitions occur later in time than the training partition. Their feature distributions may therefore differ.

The next cell uses the **standardized mean difference (SMD)**:

\[
\text{SMD} =
\frac{\mu_\text{comparison}-\mu_\text{train}}
{\sqrt{(\sigma^2_\text{comparison}+\sigma^2_\text{train})/2}}
\]

The sign indicates direction. The absolute value indicates how large the mean shift is relative to the pooled spread. SMD is a descriptive diagnostic, not a proof that the shift causes lower model performance.

When both partitions are constant, identical values have SMD 0. Different constant values are labelled `constant_value_changed`; SMD is undefined because the denominator is zero. Those cases remain visible in the table rather than being ranked as no shift.

In [ ]:
# @title 3.4 Examine feature shift, including constant-valued features
TOP_SHIFT_FEATURES = 12 # @param {type:"slider", min:5, max:17, step:1}
shift_rows=[]
for feature in feature_columns:
    for split in ("val","test"):
        stats=shift_statistics(frames["train"][feature],frames[split][feature])
        shift_rows.append({"feature":feature,"comparison":split,**stats})
shift_report=pd.DataFrame(shift_rows)
print("Special cases: a changed constant is a real change, not an SMD of zero.")
display(shift_report[~shift_report.status.isin(["defined","identical_constants"])])
display(shift_report.round(4))
finite=shift_report[shift_report.smd.notna()].copy()
finite["absolute_smd"]=finite.smd.abs()
plot_data=finite.nlargest(TOP_SHIFT_FEATURES,"absolute_smd").sort_values("absolute_smd")
if not plot_data.empty:
    plt.figure(figsize=(9,6))
    plt.barh(plot_data.feature+" / "+plot_data.comparison,plot_data.smd)
    plt.axvline(0,linewidth=1)
    plt.title("Defined feature mean shifts relative to training")
    plt.xlabel("Standardized mean difference (undefined cases shown separately)")
    plt.tight_layout();show_and_save_plot()

In [ ]:
# @title 3.5 Rank training-set feature associations with the target

CORRELATION_METHOD = "spearman" # @param ["spearman", "pearson"]
TOP_CORRELATED_FEATURES = 12 # @param {type:"slider", min:5, max:17, step:1}

training_numeric = frames["train"][
    feature_columns + [TARGET_COLUMN]
].apply(pd.to_numeric, errors="coerce")

correlations = (
    training_numeric
    .corr(method=CORRELATION_METHOD)[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
)

ordered_features = (
    correlations.abs()
    .sort_values(ascending=False)
    .head(TOP_CORRELATED_FEATURES)
    .index
)

correlation_report = (
    correlations.loc[ordered_features]
    .sort_values()
    .rename("correlation")
    .to_frame()
)

display(correlation_report.round(4))

plt.figure(figsize=(8, 6))
plt.barh(
    correlation_report.index,
    correlation_report["correlation"],
)
plt.axvline(0, linewidth=1)
plt.title(
    f"Training-set feature association with target "
    f"({CORRELATION_METHOD})"
)
plt.xlabel("Correlation")
plt.ylabel("Feature")
plt.tight_layout()
show_and_save_plot()

### Interpreting correlation

- A positive value means larger feature values tend to occur with larger target values.
- A negative value means larger feature values tend to occur with smaller target values.
- A value near zero means no strong monotonic or linear relationship was detected.
- Correlation can miss nonlinear relationships.
- Correlation does not establish that changing the feature would cause the target to change.

In [ ]:
# @title 3.6 Explore one feature in detail

FEATURE = "stockout_hours" # @param ["lag_1", "lag_7", "lag_14", "roll_7_mean", "roll_28_mean", "roll_7_std", "stockout_hours", "roll_7_stockout", "discount", "holiday_flag", "activity_flag", "precpt", "avg_temperature", "avg_humidity", "avg_wind_level", "dow", "month"]
RELATIONSHIP_SPLIT = "train" # @param ["train", "val", "test"]
FEATURE_HISTOGRAM_BINS = 30 # @param {type:"slider", min:10, max:80, step:5}
MAX_SCATTER_POINTS = 3000 # @param {type:"slider", min:500, max:7000, step:500}
QUANTILE_BINS = 10 # @param {type:"slider", min:4, max:20, step:1}

if FEATURE not in feature_columns:
    raise ValueError(f"Feature not found: {FEATURE}")

plt.figure(figsize=(9, 5))

for split in ("train", "val", "test"):
    values = pd.to_numeric(
        frames[split][FEATURE],
        errors="coerce",
    ).dropna()

    plt.hist(
        values,
        bins=FEATURE_HISTOGRAM_BINS,
        density=True,
        histtype="step",
        linewidth=2,
        label=split,
    )

plt.title(f"Distribution of {FEATURE}")
plt.xlabel(FEATURE)
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
show_and_save_plot()

if (
    RELATIONSHIP_SPLIT == "test"
    and not INCLUDE_TEST_TARGETS_IN_EDA
):
    raise ValueError(
        "Test targets are sealed for EDA. "
        "Choose train or val, or explicitly enable test-target EDA."
    )

relationship_frame = (
    frames[RELATIONSHIP_SPLIT][[FEATURE, TARGET_COLUMN]]
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

unique_values = relationship_frame[FEATURE].nunique()

if unique_values <= 12:
    grouped = (
        relationship_frame
        .groupby(FEATURE, observed=True)[TARGET_COLUMN]
        .agg(["count", "mean", "median"])
        .reset_index()
    )

    display(grouped.round(4))

    plt.figure(figsize=(9, 5))
    plt.bar(
        grouped[FEATURE].astype(str),
        grouped["mean"],
    )
    plt.title(
        f"Mean target by {FEATURE} "
        f"({RELATIONSHIP_SPLIT})"
    )
    plt.xlabel(FEATURE)
    plt.ylabel("Mean target")
    plt.tight_layout()
    show_and_save_plot()
else:
    sample_size = min(
        MAX_SCATTER_POINTS,
        len(relationship_frame),
    )

    sampled = relationship_frame.sample(
        n=sample_size,
        random_state=RANDOM_SEED,
    )

    plt.figure(figsize=(8, 5))
    plt.hexbin(
        sampled[FEATURE],
        sampled[TARGET_COLUMN],
        gridsize=35,
        mincnt=1,
    )
    plt.colorbar(label="Observation count")
    plt.title(
        f"{FEATURE} versus target "
        f"({RELATIONSHIP_SPLIT})"
    )
    plt.xlabel(FEATURE)
    plt.ylabel(TARGET_COLUMN)
    plt.tight_layout()
    show_and_save_plot()

    bins = pd.qcut(
        relationship_frame[FEATURE],
        q=min(QUANTILE_BINS, unique_values),
        duplicates="drop",
    )

    binned_summary = (
        relationship_frame
        .assign(feature_bin=bins)
        .groupby("feature_bin", observed=True)[TARGET_COLUMN]
        .agg(["count", "mean", "median"])
        .reset_index()
    )

    display(binned_summary.round(4))

    plt.figure(figsize=(9, 5))
    plt.plot(
        range(len(binned_summary)),
        binned_summary["mean"],
        marker="o",
    )
    plt.xticks(
        range(len(binned_summary)),
        binned_summary["feature_bin"].astype(str),
        rotation=45,
        ha="right",
    )
    plt.title(
        f"Mean target across {FEATURE} quantile bins "
        f"({RELATIONSHIP_SPLIT})"
    )
    plt.xlabel(f"{FEATURE} bin")
    plt.ylabel("Mean target")
    plt.tight_layout()
    show_and_save_plot()

## EDA checkpoint

Write brief answers before training a model:

1. Is the target strongly skewed?
2. How common are zero-valued targets?
3. Which feature has the strongest association with the target?
4. Which features shift most between training and validation?
5. What relationship do you observe between stockout exposure and future observed sales?
6. Which observations are descriptive associations rather than causal evidence?

**Your notes:**

- Target distribution:
- Largest feature shift:
- Strongest feature association:
- Stockout observation:
- One modeling implication:
- One limitation:

# 4. Establish a controlled evaluation protocol

## Metrics used in this activity

### Mean absolute error (MAE)

\[
\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|
\]

MAE is the average absolute error in the target's original scale. Lower is better.

### Root mean squared error (RMSE)

\[
\text{RMSE} =
\sqrt{
\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
}
\]

RMSE penalizes large errors more strongly than MAE. Lower is better.

### Median absolute error

This is the median of all absolute errors. It is less sensitive to extreme errors than MAE.

### Coefficient of determination (\(R^2\))

\(R^2\) compares squared errors with a constant predictor based on the mean of the **evaluated partition**. Higher is better, but negative values are possible when a model performs worse than that reference.

### Why MAPE is not the primary metric

Approximately 3.4% of target observations are zero-valued. Percentage error is undefined when the true value is zero and can become unstable near zero.

In [ ]:
# @title 4.1 Define shared regression metrics

def regression_metrics(
    y_true: pd.Series | np.ndarray,
    y_pred: pd.Series | np.ndarray,
) -> dict[str, float]:
    y_true_array = np.asarray(y_true, dtype=float)
    y_pred_array = np.asarray(y_pred, dtype=float)

    if y_true_array.shape != y_pred_array.shape:
        raise ValueError(
            f"Shape mismatch: y_true={y_true_array.shape}, "
            f"y_pred={y_pred_array.shape}"
        )

    if not np.isfinite(y_true_array).all():
        raise ValueError("y_true contains a non-finite value.")

    if not np.isfinite(y_pred_array).all():
        raise ValueError("y_pred contains a non-finite value.")

    return {
        "mae": float(mean_absolute_error(y_true_array, y_pred_array)),
        "rmse": float(
            np.sqrt(mean_squared_error(y_true_array, y_pred_array))
        ),
        "median_absolute_error": float(
            median_absolute_error(y_true_array, y_pred_array)
        ),
        "r2": float(r2_score(y_true_array, y_pred_array)),
    }

def make_result_record(
    *,
    model_key: str,
    model: str,
    family: str,
    condition: str,
    partition: str,
    y_true: pd.Series | np.ndarray,
    y_pred: pd.Series | np.ndarray,
    runtime_seconds: float | None = None,
    effective_mode: str | None = None,
    licence: str | None = None,
    notes: str | None = None,
) -> dict[str, Any]:
    record = {
        "model_key": model_key,
        "model": model,
        "family": family,
        "condition": condition,
        "partition": partition,
        **regression_metrics(y_true, y_pred),
        "runtime_seconds": runtime_seconds,
        "effective_mode": effective_mode,
        "licence": licence,
        "notes": notes,
        "device": "cpu",
    }
    return record

print("Metric functions are ready.")

## Baseline ladder

A meaningful experiment should not compare a foundation model only with a weak constant predictor. This notebook uses a baseline ladder:

1. **Training mean:** the same prediction for every row;
2. **Rolling-seven-day mean:** a domain-informed historical baseline;
3. **Ridge regression:** a regularized linear model;
4. **Random Forest:** a nonlinear tree ensemble; and
5. **LightGBM:** a gradient-boosted tree model.

A foundation model adds practical value only if it improves on relevant alternatives under the same split and metrics.

In [ ]:
# @title 4.2 Fit, validate, and retain classical baseline artifacts
RF_TREES = 300 # @param {type:"slider", min:100, max:1000, step:100}
LGBM_ESTIMATORS = 500 # @param {type:"slider", min:100, max:1500, step:100}
LGBM_LEARNING_RATE = 0.05 # @param {type:"number"}
assert_current_data();SESSION.assert_development()


def classical_parameters(key):
    if key.startswith("training_mean"):return {"method":"training_mean"}
    if key.startswith("rolling_7_mean"):return {"method":"rolling_7_mean"}
    if key.startswith("ridge"):return {"alpha":1.0}
    if key.startswith("random_forest"):return {"n_estimators":RF_TREES,"min_samples_leaf":2,"random_state":RANDOM_SEED}
    if key.startswith("lightgbm"):return {"n_estimators":LGBM_ESTIMATORS,"learning_rate":LGBM_LEARNING_RATE,
                                        "num_leaves":31,"colsample_bytree":0.9,"random_state":RANDOM_SEED}
    raise KeyError(key)

def classical_controls(key, features=None):
    return {**base_controls(features),"backend":"classical","parameters":classical_parameters(key),
            "software_versions":SOFTWARE_VERSIONS,"adapter_version":"classical-workshop-v2"}

model_templates={
 "ridge":Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("model",Ridge(alpha=1.0))]),
 "random_forest":Pipeline([("imputer",SimpleImputer(strategy="median")),
                  ("model",RandomForestRegressor(**classical_parameters("random_forest"),n_jobs=-1))])}
if LIGHTGBM_AVAILABLE:
    model_templates["lightgbm"]=Pipeline([("imputer",SimpleImputer(strategy="median")),
                               ("model",LGBMRegressor(**classical_parameters("lightgbm"),n_jobs=-1,verbosity=-1))])
models={"training_mean":ReferenceRegressor(y_train.mean()),
        "rolling_7_mean":ReferenceRegressor(y_train.mean(),"roll_7_mean"),
        **{k:clone(v) for k,v in model_templates.items()}}
names={"training_mean":"Training Mean","rolling_7_mean":"Rolling 7-Day Mean","ridge":"Ridge Regression",
       "random_forest":"Random Forest","lightgbm":"LightGBM"}
fitted_models={};CLASSICAL_RECEIPTS=[];validation_predictions={}
for key,model in models.items():
    start=time.perf_counter()
    if key in model_templates:model.fit(X_train,y_train)
    elapsed=time.perf_counter()-start
    ctrl=classical_controls(key)
    identity={"experiment_id":SESSION.experiment_id,"model_key":key,"controls":ctrl,
              "split_sha256":DATASET_IDENTITY["split_sha256"],"features":feature_columns}
    receipt=save_classical_run(SESSION,key,model,frames["train"],frames["val"],ctrl,fit_seconds=elapsed,
               identity=identity,model_name=names[key],family="Naive baseline" if key not in model_templates else "Classical ML",
               mode="fixed" if key not in model_templates else "trained from scratch")
    CLASSICAL_RECEIPTS.append(receipt);fitted_models[key]=model
    validation_predictions[key]=np.asarray(model.predict(X_val),dtype=float)
CURRENT_CLASSICAL_KEYS=list(models)
baseline_validation_results,_=load_results(CLASSICAL_RECEIPTS,frames["val"])
baseline_records=baseline_validation_results.to_dict("records")
display(baseline_validation_results[["model","mae","rmse","median_absolute_error","r2","runtime_seconds"]].round(5))

In [ ]:
# @title 4.3 Visualize baseline validation performance

PLOT_METRIC = "mae" # @param ["mae", "rmse", "median_absolute_error", "r2"]

plot_results = baseline_validation_results.sort_values(
    PLOT_METRIC,
    ascending=(PLOT_METRIC != "r2"),
)

plt.figure(figsize=(9, 5))
plt.barh(
    plot_results["model"],
    plot_results[PLOT_METRIC],
)
plt.title(f"Classical baseline performance on validation data: {PLOT_METRIC}")
plt.xlabel(PLOT_METRIC)
plt.ylabel("Model")
plt.tight_layout()
show_and_save_plot(run_ids=plot_results.run_id.dropna().tolist())

In [ ]:
# @title 4.4 Inspect one baseline's predictions

BASELINE_TO_INSPECT = "" # @param {type:"string"}
MAX_POINTS = 1600 # @param {type:"slider", min:200, max:1600, step:200}

BASELINE_TO_INSPECT=choose_diagnostic_model(validation_predictions,BASELINE_TO_INSPECT)
if BASELINE_TO_INSPECT not in validation_predictions:
    raise ValueError(
        f"No validation predictions found for {BASELINE_TO_INSPECT}. "
        "Choose a model that was fitted successfully."
    )

inspection = pd.DataFrame(
    {
        "observed": y_val.to_numpy(),
        "predicted": validation_predictions[BASELINE_TO_INSPECT],
    }
)

if len(inspection) > MAX_POINTS:
    inspection = inspection.sample(
        n=MAX_POINTS,
        random_state=RANDOM_SEED,
    )

lower = float(
    min(inspection["observed"].min(), inspection["predicted"].min())
)
upper = float(
    max(inspection["observed"].max(), inspection["predicted"].max())
)

plt.figure(figsize=(7, 6))
plt.scatter(
    inspection["observed"],
    inspection["predicted"],
    alpha=0.5,
)
plt.plot([lower, upper], [lower, upper], linestyle="--")
plt.title(f"Observed versus predicted: {BASELINE_TO_INSPECT}")
plt.xlabel("Observed target")
plt.ylabel("Predicted target")
plt.tight_layout()
show_and_save_plot(run_ids=baseline_validation_results.loc[baseline_validation_results.model_key.eq(BASELINE_TO_INSPECT),"run_id"].tolist())

residuals = inspection["observed"] - inspection["predicted"]

plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=40)
plt.axvline(0, linewidth=1)
plt.title(f"Residual distribution: {BASELINE_TO_INSPECT}")
plt.xlabel("Residual = observed - predicted")
plt.ylabel("Count")
plt.tight_layout()
show_and_save_plot(run_ids=baseline_validation_results.loc[baseline_validation_results.model_key.eq(BASELINE_TO_INSPECT),"run_id"].tolist())

## Baseline checkpoint

Answer the following:

1. Which baseline has the lowest validation MAE?
2. Does the same model also have the lowest RMSE?
3. Is its \(R^2\) positive?
4. Does one model make a small number of very large errors?
5. Does a complex classical model clearly improve on the rolling-seven-day mean?

Do not open the test metrics yet. The validation results are sufficient for model comparison at this stage.

# 5. Run the tabular foundation models locally in Colab

The regressors in this section execute inside the notebook. You do **not** submit model jobs through DIMER.

The notebook uses the same pinned pipeline repositories and model identities represented by the DIMER profiles. Each repository is checked out at a fixed commit, installed in its own virtual environment, and invoked through its public pipeline API.

## Why use isolated environments?

The four model stacks pin different versions of PyTorch, NumPy, pandas, scikit-learn, Hugging Face Hub, and other packages. Installing them all into the main notebook environment could replace packages that the EDA and classical-baseline sections already use.

Isolation provides three advantages:

1. each model receives the dependency versions declared by its own repository;
2. one model's installation cannot silently change another model's runtime; and
3. the notebook can record the exact repository commit and package set used for each result.

## Public notebook capabilities

- **Mitra:** in-context learning; optional GPU fine-tuning
- **TabDPT:** in-context learning only
- **TabPFN-3:** in-context learning only in the public pipeline; v3 weights are non-commercial
- **TabICLv2:** in-context learning; optional GPU fine-tuning

The first environment build can be slow and storage-intensive. Select fewer models when workshop time or runtime storage is limited.

In [ ]:
# @title 5.1 Select the foundation-model conditions

RUN_MITRA_ICL = True # @param {type:"boolean"}
RUN_TABDPT_ICL = True # @param {type:"boolean"}
RUN_TABPFN3_ICL = True # @param {type:"boolean"}
RUN_TABICLV2_ICL = True # @param {type:"boolean"}

RUN_MITRA_FINETUNED = False # @param {type:"boolean"}
RUN_TABICLV2_FINETUNED = False # @param {type:"boolean"}

DEVICE_PREFERENCE = "auto" # @param ["auto", "cpu", "cuda"]
REBUILD_MODEL_ENVIRONMENTS = False # @param {type:"boolean"}
FORCE_MODEL_RERUN = False # @param {type:"boolean"}
# Fitted artifacts are always retained until the runtime ends; final testing reloads them.

MITRA_TIME_LIMIT_SECONDS = 600 # @param {type:"integer"}
MITRA_FINE_TUNE_STEPS = 50 # @param {type:"slider", min:10, max:100, step:10}

TABDPT_N_ENSEMBLES = 4 # @param {type:"slider", min:1, max:16, step:1}
TABDPT_CONTEXT_SIZE = 4180 # @param {type:"integer"}
TABDPT_BATCH_SIZE = 4096 # @param {type:"integer"}

TABPFN_N_ESTIMATORS = 4 # @param {type:"slider", min:1, max:16, step:1}
TABICL_N_ESTIMATORS = 8 # @param {type:"slider", min:1, max:32, step:1}
TABICL_FINE_TUNE_EPOCHS = 10 # @param {type:"slider", min:1, max:30, step:1}
TABICL_FINE_TUNE_TIME_LIMIT = 600 # @param {type:"integer"}
TABICL_FINE_TUNE_PATIENCE = 3 # @param {type:"integer"}

MODEL_REPOSITORIES = {
    "mitra": {
        "repository": "kurtvalcorza/mitra-regressor-pipeline",
        "commit": "a2b36c21fe92d9e42e2f3310c98e9e1c7fe8d7af",
        "import_name": "mitra_pipeline",
        "licence": "Apache-2.0",
    },
    "tabdpt": {
        "repository": "kurtvalcorza/tabdpt-regressor-pipeline",
        "commit": "21085dd2e148041d8fa88944c21532b4bdf9a7ea",
        "import_name": "tabdpt_regressor_pipeline",
        "licence": "Apache-2.0",
    },
    "tabpfn": {
        "repository": "kurtvalcorza/tabpfn-regressor-pipeline",
        "commit": "45dddb070326b9994a18e689add0546d30684239",
        "import_name": "tabpfn_regressor_pipeline",
        "licence": "TabPFN-3 licence v1.0 — non-commercial weights",
    },
    "tabicl": {
        "repository": "kurtvalcorza/tabicl-regressor-pipeline",
        "commit": "08146a323d516fb8dc9aac7f51ad17994e37294c",
        "import_name": "tabicl_regressor_pipeline",
        "licence": "BSD-3-Clause",
    },
}

CONDITION_SPECS = {
    "mitra_icl": {
        "environment": "mitra",
        "model_family": "mitra",
        "display_name": "Mitra Regressor",
        "condition": "in_context",
        "configuration": {
            "fine_tune": False,
            "time_limit_seconds": MITRA_TIME_LIMIT_SECONDS,
            "fine_tune_steps": 0,
        },
    },
    "mitra_finetuned": {
        "environment": "mitra",
        "model_family": "mitra",
        "display_name": "Mitra Regressor",
        "condition": "fine_tuned",
        "configuration": {
            "fine_tune": True,
            "time_limit_seconds": MITRA_TIME_LIMIT_SECONDS,
            "fine_tune_steps": MITRA_FINE_TUNE_STEPS,
        },
    },
    "tabdpt_icl": {
        "environment": "tabdpt",
        "model_family": "tabdpt",
        "display_name": "TabDPT v1.2 Regressor",
        "condition": "in_context",
        "configuration": {
            "n_ensembles": TABDPT_N_ENSEMBLES,
            "context_size": TABDPT_CONTEXT_SIZE,
            "batch_size": TABDPT_BATCH_SIZE,
        },
    },
    "tabpfn3_icl": {
        "environment": "tabpfn",
        "model_family": "tabpfn",
        "display_name": "TabPFN-3 Regressor",
        "condition": "in_context",
        "configuration": {
            "n_estimators": TABPFN_N_ESTIMATORS,
        },
    },
    "tabiclv2_icl": {
        "environment": "tabicl",
        "model_family": "tabicl",
        "display_name": "TabICLv2 Regressor",
        "condition": "in_context",
        "configuration": {
            "n_estimators": TABICL_N_ESTIMATORS,
        },
    },
    "tabiclv2_finetuned": {
        "environment": "tabicl",
        "model_family": "tabicl",
        "display_name": "TabICLv2 Regressor",
        "condition": "fine_tuned",
        "configuration": {
            "n_estimators": TABICL_N_ESTIMATORS,
            "fine_tune_epochs": TABICL_FINE_TUNE_EPOCHS,
            "fine_tune_time_limit": TABICL_FINE_TUNE_TIME_LIMIT,
            "fine_tune_patience": TABICL_FINE_TUNE_PATIENCE,
        },
    },
}

selected_foundation_conditions = []

if RUN_MITRA_ICL:
    selected_foundation_conditions.append("mitra_icl")
if RUN_TABDPT_ICL:
    selected_foundation_conditions.append("tabdpt_icl")
if RUN_TABPFN3_ICL:
    selected_foundation_conditions.append("tabpfn3_icl")
if RUN_TABICLV2_ICL:
    selected_foundation_conditions.append("tabiclv2_icl")
if RUN_MITRA_FINETUNED:
    selected_foundation_conditions.append("mitra_finetuned")
if RUN_TABICLV2_FINETUNED:
    selected_foundation_conditions.append("tabiclv2_finetuned")

run_plan_rows = []

for condition_key in selected_foundation_conditions:
    condition = CONDITION_SPECS[condition_key]
    repository = MODEL_REPOSITORIES[condition["environment"]]

    run_plan_rows.append(
        {
            "model_key": condition_key,
            "model": condition["display_name"],
            "condition": condition["condition"],
            "repository": repository["repository"],
            "repository_commit": repository["commit"],
            "weights_licence": repository["licence"],
            "device_preference": DEVICE_PREFERENCE,
            "configuration": json.dumps(
                condition["configuration"],
                sort_keys=True,
            ),
        }
    )

foundation_run_plan = pd.DataFrame(run_plan_rows)

if foundation_run_plan.empty:
    print("No foundation-model conditions selected.")
else:
    display(foundation_run_plan)
# Immutable model provenance retained alongside the pinned repository revision.
MODEL_PROVENANCE={
 "mitra":{"model_id":"autogluon/mitra-regressor","revision":"5f277aa8f69042d39d6ac3612aed18bb9279bd95",
          "weights_sha256":"d8e75c62af0bec2fd404b0ad20a442d951d43ca6d331315cfcc0509b54f2c642"},
 "tabdpt":{"model_id":"Layer6/TabDPT","revision":"4462ffbd1d8dea25d4862d30beed4b70cd596ae5",
           "weights_sha256":"06680220fd66c4524051706b98c1c659a674d19d3a766cd0bb276505e99faccd"},
 "tabpfn":{"model_id":"Prior-Labs/tabpfn_3","revision":"24a16a89d245878b846555110985634aa2e656d7",
           "weights_sha256":"311ce18d97e9533d8585eaadafe040fbdd8070533209ed8696641dadc97a7301"},
 "tabicl":{"model_id":"jingang/TabICL","revision":"4dcd344ece2c00be9e831fdd35bed57b5ad83e19",
           "weights_sha256":"0db9cb538f114e79026bf08f45f41ad8dd7ad2de2aaca9a5ca8cd3bd9748ae7a"}}
CURRENT_FOUNDATION_KEYS=list(selected_foundation_conditions)


## 5.2 Build isolated model environments

This step performs four operations for each selected model family:

1. clone the pipeline repository at the fixed commit listed in the run plan;
2. ensure the `uv` environment manager is available;
3. create a dedicated **Python 3.12** virtual environment, downloading a managed Python 3.12 interpreter when the host runtime does not provide one; and
4. install the repository and the runtime dependencies declared in its `pyproject.toml`.

The environments are cached for the life of the Colab runtime. Re-running the cell normally reuses them. Enable **Rebuild model environments** only when an installation is damaged or you deliberately need a clean rebuild.

The installation logs are saved under:

```text
/content/dimer_tabular_workshop/install_logs/
```

This avoids the previous failure mode where a Python 3.13 notebook kernel caused all four foundation models to be silently skipped. The notebook kernel itself is not replaced; each foundation model runs in its own compatible Python 3.12 subprocess environment.

All rebuilds use fresh directories rather than deleting previous environments. Do not call environment preparation after freezing.


In [ ]:
# @title 5.2 Build or reuse the selected model environments

WORKSHOP_ROOT = WORKSPACE_ROOT
REPOSITORY_ROOT = WORKSHOP_ROOT / "repositories"
ENVIRONMENT_ROOT = WORKSHOP_ROOT / "environments"
RUNNER_ROOT = SUPPORT_DIR
FOUNDATION_OUTPUT_ROOT = SESSION_ROOT / "runs"
INSTALL_LOG_ROOT = WORKSHOP_ROOT / "install_logs"

for directory in (
    WORKSHOP_ROOT,
    REPOSITORY_ROOT,
    ENVIRONMENT_ROOT,
    RUNNER_ROOT,
    FOUNDATION_OUTPUT_ROOT,
    INSTALL_LOG_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

def shell_join(command: list[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)

def stream_command(
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    log_path: Path | None = None,
) -> None:
    print(f"$ {shell_join(command)}")

    merged_env = os.environ.copy()
    merged_env.update(
        {
            "PIP_DISABLE_PIP_VERSION_CHECK": "1",
            "PIP_PROGRESS_BAR": "off",
            "PYTHONUNBUFFERED": "1",
        }
    )

    if env:
        merged_env.update(env)

    log_handle = (
        log_path.open("a", encoding="utf-8")
        if log_path is not None
        else None
    )

    try:
        process = subprocess.Popen(
            [str(part) for part in command],
            cwd=str(cwd) if cwd else None,
            env=merged_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None

        for line in process.stdout:
            print(line, end="")
            if log_handle is not None:
                log_handle.write(line)

        return_code = process.wait()

        if return_code != 0:
            raise subprocess.CalledProcessError(
                return_code,
                command,
            )
    finally:
        if log_handle is not None:
            log_handle.close()

FOUNDATION_PYTHON_SPEC = "3.12"

def ensure_uv() -> Path:
    uv_path = shutil.which("uv")
    if uv_path:
        return Path(uv_path)

    bootstrap_log = INSTALL_LOG_ROOT / "uv-bootstrap.log"
    print("uv is not available; installing it into the notebook host environment ...")
    stream_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "uv",
        ],
        log_path=bootstrap_log,
    )

    uv_path = shutil.which("uv")
    if not uv_path:
        candidate = Path(sys.executable).resolve().parent / "uv"
        if candidate.exists():
            uv_path = str(candidate)

    if not uv_path:
        raise RuntimeError(
            "uv installation completed but the executable could not be located."
        )

    return Path(uv_path)

def environment_python(environment_path: Path) -> Path:
    if os.name == "nt":
        return environment_path / "Scripts" / "python.exe"
    return environment_path / "bin" / "python"

def repository_path(environment_key: str) -> Path:
    return REPOSITORY_ROOT / (environment_key+"-"+MODEL_REPOSITORIES[environment_key]["commit"][:12])

def environment_path(environment_key: str) -> Path:
    return ENVIRONMENT_ROOT / (environment_key+"-"+MODEL_REPOSITORIES[environment_key]["commit"][:12])

def checked_out_commit(path: Path) -> str | None:
    if not (path / ".git").exists():
        return None

    completed = subprocess.run(
        ["git", "-C", str(path), "rev-parse", "HEAD"],
        check=False,
        capture_output=True,
        text=True,
    )

    if completed.returncode != 0:
        return None

    return completed.stdout.strip()

def ensure_repository(environment_key: str) -> Path:
    specification = MODEL_REPOSITORIES[environment_key]
    destination = repository_path(environment_key)
    expected_commit = specification["commit"]

    if (
        destination.exists()
        and checked_out_commit(destination) != expected_commit
    ):
        raise RuntimeError(f"Refusing to delete existing directory: {destination}. Use a fresh runtime or inspect it manually.")

    if not destination.exists():
        repository_url = (
            "https://github.com/"
            f"{specification['repository']}.git"
        )

        stream_command(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                repository_url,
                str(destination),
            ]
        )
        stream_command(
            [
                "git",
                "-C",
                str(destination),
                "fetch",
                "--depth",
                "1",
                "origin",
                expected_commit,
            ]
        )
        stream_command(
            [
                "git",
                "-C",
                str(destination),
                "checkout",
                "--detach",
                expected_commit,
            ]
        )

    actual_commit = checked_out_commit(destination)

    if actual_commit != expected_commit:
        raise RuntimeError(
            f"{environment_key}: expected repository commit "
            f"{expected_commit}, found {actual_commit}"
        )

    return destination

def ensure_environment(environment_key: str) -> Path:
    repository = ensure_repository(environment_key)
    destination = environment_path(environment_key)
    python_path = environment_python(destination)
    marker_path = destination / ".workshop_environment.json"
    expected_commit = MODEL_REPOSITORIES[
        environment_key
    ]["commit"]

    marker = None

    if marker_path.exists():
        try:
            marker = json.loads(
                marker_path.read_text(encoding="utf-8")
            )
        except Exception:
            marker = None

    reusable = (
        not REBUILD_MODEL_ENVIRONMENTS
        and python_path.exists()
        and isinstance(marker, dict)
        and marker.get("repository_commit") == expected_commit
        and marker.get("python_spec") == FOUNDATION_PYTHON_SPEC
    )

    if reusable:
        print(
            f"Reusing {environment_key} environment: "
            f"{destination}"
        )
        return python_path

    if destination.exists():
        destination=new_owned_directory(ENVIRONMENT_ROOT,environment_key+"-rebuild")
        marker_path=destination / ".workshop_environment.json"

    print(
        f"\nCreating Python {FOUNDATION_PYTHON_SPEC} environment "
        f"for {environment_key} ..."
    )
    install_log = INSTALL_LOG_ROOT / f"{environment_key}.log"
    uv_path = ensure_uv()
    stream_command(
        [
            str(uv_path),
            "venv",
            "--seed",
            "--python",
            FOUNDATION_PYTHON_SPEC,
            str(destination),
        ],
        log_path=install_log,
    )

    python_path = environment_python(destination)
    if not python_path.exists():
        raise RuntimeError(
            f"uv did not create the expected interpreter: {python_path}"
        )

    pip_check = subprocess.run(
        [str(python_path), "-m", "pip", "--version"],
        capture_output=True,
        text=True,
    )
    if pip_check.returncode != 0:
        stream_command(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--ignore-installed",
                f"--prefix={destination}",
                "pip",
                "setuptools",
                "wheel",
            ],
            log_path=install_log,
        )

    try:
        stream_command(
            [
                str(python_path),
                "-m",
                "pip",
                "install",
                "--upgrade",
                "pip",
                "setuptools",
                "wheel",
            ],
            log_path=install_log,
        )
    except subprocess.CalledProcessError as err:
        print(f"Warning: pip/setuptools/wheel upgrade returned non-zero ({err}); proceeding.")

    stream_command(
        [
            str(python_path),
            "-m",
            "pip",
            "install",
            "--editable",
            str(repository),
        ],
        log_path=install_log,
    )

    import_name = MODEL_REPOSITORIES[
        environment_key
    ]["import_name"]

    stream_command(
        [
            str(python_path),
            "-c",
            (
                f"import {import_name}; "
                f"print('imported {import_name}')"
            ),
        ],
        log_path=install_log,
    )

    freeze = subprocess.check_output(
        [
            str(python_path),
            "-m",
            "pip",
            "freeze",
        ],
        text=True,
    )

    (
        destination / "pip-freeze.txt"
    ).write_text(freeze, encoding="utf-8")

    marker_path.write_text(
        json.dumps(
            {
                "repository": MODEL_REPOSITORIES[
                    environment_key
                ]["repository"],
                "repository_commit": expected_commit,
                "python_spec": FOUNDATION_PYTHON_SPEC,
                "python": str(python_path),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    return python_path

if selected_foundation_conditions:
    print(
        f"Notebook host Python: {sys.version.split()[0]}. "
        f"Foundation-model environments will use isolated Python {FOUNDATION_PYTHON_SPEC} runtimes."
    )

selected_environment_keys = sorted(
    {
        CONDITION_SPECS[key]["environment"]
        for key in selected_foundation_conditions
    }
)

MODEL_ENVIRONMENT_PYTHONS = {}

def describe_environment(environment_key):
    py = MODEL_ENVIRONMENT_PYTHONS[environment_key]
    repo = repository_path(environment_key)
    expected = MODEL_REPOSITORIES[environment_key]["commit"]
    if checked_out_commit(repo) != expected:
        raise ValueError("Repository revision changed.")
    subprocess.run(["git", "-C", str(repo), "diff", "--quiet", "HEAD", "--"], check=True)
    script = "import importlib.metadata as m,json,platform,torch;print(json.dumps({'python':platform.python_version(),'cuda_available':torch.cuda.is_available(),'cuda_device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'packages':sorted((d.metadata['Name'],d.version) for d in m.distributions())},sort_keys=True))"
    observed = json.loads(subprocess.check_output([str(py), "-c", script], text=True))
    return {"python": py, "repository_path": repo, "signature": fingerprint(observed), "runtime": observed}

ENVIRONMENT_RECORDS = {}
ENVIRONMENT_ERRORS = {}

if selected_environment_keys:
    disk = shutil.disk_usage(WORKSPACE_ROOT)
    print(
        "Free runtime storage before installation: "
        f"{disk.free / 1024**3:.1f} GiB"
    )

    for environment_key in selected_environment_keys:
        try:
            MODEL_ENVIRONMENT_PYTHONS[
                environment_key
            ] = ensure_environment(environment_key)
            ENVIRONMENT_RECORDS[
                environment_key
            ] = describe_environment(environment_key)
        except Exception as err:
            ENVIRONMENT_ERRORS[environment_key] = (
                f"{type(err).__name__}: {err}"
            )
            print(
                f"\nERROR preparing {environment_key}: "
                f"{ENVIRONMENT_ERRORS[environment_key]}"
            )

    if MODEL_ENVIRONMENT_PYTHONS:
        print("\nReady foundation-model environments:")
        display(
            pd.Series(
                {
                    key: str(value)
                    for key, value in MODEL_ENVIRONMENT_PYTHONS.items()
                    if key in ENVIRONMENT_RECORDS
                },
                name="python",
            ).to_frame()
        )

    if ENVIRONMENT_ERRORS:
        print("\nEnvironment failures:")
        display(
            pd.Series(
                ENVIRONMENT_ERRORS,
                name="error",
            ).to_frame()
        )

    requested_foundation_conditions = list(selected_foundation_conditions)
    selected_foundation_conditions = [
        key
        for key in requested_foundation_conditions
        if CONDITION_SPECS[key]["environment"] in ENVIRONMENT_RECORDS
    ]
    CURRENT_FOUNDATION_KEYS = list(selected_foundation_conditions)

    unavailable_conditions = [
        key
        for key in requested_foundation_conditions
        if key not in selected_foundation_conditions
    ]
    if unavailable_conditions:
        print(
            "Conditions unavailable after environment preparation:",
            unavailable_conditions,
        )

    if requested_foundation_conditions and not selected_foundation_conditions:
        raise RuntimeError(
            "No selected foundation-model environment could be prepared. "
            "See the environment failure table above; the notebook will not silently continue as a baseline-only comparison."
        )
else:
    print("No foundation-model conditions selected.")


## 5.3 Create the local model runner
This bundled code loads the pinned weights, fits or conditions each regressor, and retains its fitted artifact. It also reloads that artifact and checks that validation predictions agree before a run is accepted.

Every prediction CSV uses **`row_id`, `target`, `prediction`**. A row ID is an index within one supplied split, not a store/product identifier.

**Mitra validation:** this notebook uses a small, explicitly identified adapter overlay. It calls AutoGluon with `tuning_data=val.csv`, disables bagging, stacking and full refitting, and checks the observed training/validation row counts. The pinned repository's weight loader remains unchanged. Validation therefore supports selection/early stopping; it is not an untouched test. Per-query internal context size is not assumed identical across model families.

Local pickle/native model files are trusted only because this notebook produced them in the current experiment and verifies their registered hashes. Do not replace them with external artifacts.

In [ ]:
# @title 5.3 Write the local model runner
RUNNER_SCRIPT=SUPPORT_DIR / "local_foundation_regressor.py"
RUNNER_SOURCE='from __future__ import annotations\nimport argparse, importlib.metadata, json, math, os, pickle, platform, time\nfrom pathlib import Path\nfrom typing import Any\nimport numpy as np\nimport pandas as pd\nfrom workshop_core import (sha256_file, write_json, validate_receipt, prediction_frame,\n                           canonical_predictions, regression_metrics,\n                           fit_mitra_with_chronological_validation)\n\ndef seed_all(seed):\n    import random, torch\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n\ndef package_version(name):\n    try: return importlib.metadata.version(name)\n    except importlib.metadata.PackageNotFoundError: return "not installed"\n\ndef resolve_device(preference):\n    import torch\n    if preference == "cpu": return "cpu"\n    if preference == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA requested but unavailable; use in-context CPU or select a GPU runtime.")\n    return "cuda" if torch.cuda.is_available() else "cpu"\n\ndef artifact_metadata(output_dir, kind, features, device, **extra):\n    root=output_dir / "artifact"; root.mkdir(parents=True, exist_ok=True)\n    meta={"format":kind, "features":list(features), "device":device, **extra}\n    write_json(root / "metadata.json",meta)\n    return meta\n\ndef save_pickle_artifact(output_dir, payload, features, device, **extra):\n    meta=artifact_metadata(output_dir,"trusted_local_pickle",features,device,**extra)\n    with (output_dir / "artifact" / "fitted.pkl").open("wb") as f:\n        pickle.dump(payload,f,protocol=pickle.HIGHEST_PROTOCOL)\n    return meta\n\ndef load_predictor(root, device):\n    # Caller verifies registered receipt and all artifact hashes BEFORE entering here.\n    meta=json.loads((root / "artifact" / "metadata.json").read_text())\n    if device != meta["device"]:\n        raise RuntimeError("Frozen model device differs. Use the original runtime; do not refit silently.")\n    if meta["format"] == "autogluon":\n        from autogluon.tabular import TabularPredictor\n        model=TabularPredictor.load(str(root / "artifact" / "autogluon"))\n        return lambda X: np.asarray(model.predict(X[meta["features"]]), dtype=float)\n    if meta["format"] == "tabpfn_native":\n        from tabpfn_regressor_pipeline.pipeline import TabPFNRegressorPipeline\n        model=TabPFNRegressorPipeline.from_artifact(root / "artifact" / "tabpfn", device=device)\n        return lambda X: model.predict_values(X[meta["features"]])\n    if meta["format"] != "trusted_local_pickle":\n        raise ValueError("Unknown fitted artifact format.")\n    with (root / "artifact" / "fitted.pkl").open("rb") as f: payload=pickle.load(f)\n    model=payload["model"]\n    if "encoders" in payload:\n        from tabicl_regressor_pipeline.api import apply_categorical_encoder\n        def predict(X):\n            encoded,_=apply_categorical_encoder(X[meta["features"]],payload["encoders"])\n            return model.predict(encoded)\n        return predict\n    return lambda X: model.predict(X[meta["features"]], **payload.get("predict_kwargs",{}))\n\n\ndef run_mitra(config, train, validation, test, output_dir, device):\n    from mitra_pipeline.tutorial_api import (MODEL_ID, MODEL_KEY, MODEL_LICENSE, MODEL_REVISION,\n                                             MitraRegressionPipeline)\n    repository=Path(config["repository_path"])\n    weights=repository / "weights" / MODEL_KEY\n    settings=config["model_configuration"]\n    fine_tune=bool(settings.get("fine_tune",False))\n    if fine_tune and device != "cuda": raise RuntimeError("Mitra fine-tuning requires CUDA.")\n    start=time.perf_counter()\n    verified=MitraRegressionPipeline.from_pretrained(weights_dir=weights,allow_download=True,\n                  hf_home=repository / ".workshop-hf-cache",device=device)\n    predictor,counts=fit_mitra_with_chronological_validation(train,validation,\n                  path=output_dir / "artifact" / "autogluon", fine_tune=fine_tune,\n                  steps=int(settings.get("fine_tune_steps",0)),\n                  time_limit=int(settings.get("time_limit_seconds",600)),\n                  seed=int(config["seed"]),device=device)\n    fit_seconds=time.perf_counter()-start\n    features=[c for c in train if c != "target"]\n    seed_all(config["seed"])\n    start=time.perf_counter()\n    pred=np.asarray(predictor.predict(validation[features]),dtype=float)\n    predict_seconds=time.perf_counter()-start\n    artifact_metadata(output_dir,"autogluon",features,device)\n    return {"model_id":MODEL_ID,"model_revision":MODEL_REVISION,"model_licence":MODEL_LICENSE,\n            "effective_mode":"fine_tuned" if fine_tune else "in_context","device":device,\n            "runtime_seconds":{"fit":fit_seconds,"validation_prediction":predict_seconds},\n            "training_audit":counts,"adapter_policy":"workshop-chronological-mitra-v2"},pred,None\n\n\ndef run_tabdpt(\n    config: dict[str, Any],\n    train: pd.DataFrame,\n    validation: pd.DataFrame,\n    test: pd.DataFrame | None,\n    output_dir: Path,\n    device: str,\n) -> tuple[dict[str, Any], np.ndarray, np.ndarray | None]:\n    from tabdpt_regressor_pipeline.pipeline import (\n        MODEL_ID,\n        MODEL_KEY,\n        MODEL_LICENSE,\n        MODEL_REVISION,\n        TabDPTRegressionPipeline,\n    )\n\n    repository = Path(config["repository_path"])\n    weights_dir = repository / "weights" / MODEL_KEY\n    settings = config["model_configuration"]\n\n    start = time.perf_counter()\n\n    pipeline = TabDPTRegressionPipeline.from_pretrained(\n        weights_dir=weights_dir,\n        allow_download=True,\n        device=device,\n        seed=int(config["seed"]),\n        verbose=False,\n        use_flash=False,\n    )\n\n    pipeline.fit(\n        train,\n        target_column="target",\n        seed=int(config["seed"]),\n    )\n\n    fit_seconds = time.perf_counter() - start\n    feature_columns = [\n        column\n        for column in train.columns\n        if column != "target"\n    ]\n\n    prediction_kwargs = {\n        "n_ensembles": int(\n            settings.get("n_ensembles", 4)\n        ),\n        "context_size": min(\n            int(settings.get("context_size", len(train))),\n            len(train),\n        ),\n        "batch_size": int(\n            settings.get("batch_size", 4096)\n        ),\n        "seed": int(config["seed"]),\n    }\n\n    start = time.perf_counter()\n    validation_prediction = pipeline.predict(\n        validation[feature_columns],\n        **prediction_kwargs,\n    ).to_numpy()\n    validation_prediction_seconds = (\n        time.perf_counter() - start\n    )\n\n    test_prediction = None\n    test_prediction_seconds = None\n\n    if test is not None:\n        start = time.perf_counter()\n        test_prediction = pipeline.predict(\n            test[feature_columns],\n            **prediction_kwargs,\n        ).to_numpy()\n        test_prediction_seconds = (\n            time.perf_counter() - start\n        )\n\n    result = {\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n        "model_licence": MODEL_LICENSE,\n        "effective_mode": "in_context",\n        "device": device,\n        "runtime_seconds": {\n            "fit": fit_seconds,\n            "validation_prediction": validation_prediction_seconds,\n            "test_prediction": test_prediction_seconds,\n        },\n        "software_versions": {\n            "python": platform.python_version(),\n            "tabdpt": package_version("tabdpt"),\n            "torch": package_version("torch"),\n            "pandas": package_version("pandas"),\n        },\n    }\n\n    save_pickle_artifact(output_dir, {"model":pipeline,"predict_kwargs":prediction_kwargs},\n                         feature_columns,device)\n    result["training_audit"]={"train_rows_supplied":len(train),\n        "registered_support_rows":len(train),"context_limit_per_ensemble":prediction_kwargs["context_size"]}\n    return result, validation_prediction, test_prediction\n\ndef run_tabpfn(\n    config: dict[str, Any],\n    train: pd.DataFrame,\n    validation: pd.DataFrame,\n    test: pd.DataFrame | None,\n    output_dir: Path,\n    device: str,\n) -> tuple[dict[str, Any], np.ndarray, np.ndarray | None]:\n    from tabpfn_regressor_pipeline.pipeline import (\n        MODEL_ID,\n        MODEL_KEY,\n        MODEL_LICENSE,\n        MODEL_REVISION,\n        TabPFNRegressorPipeline,\n    )\n\n    repository = Path(config["repository_path"])\n    weights_dir = repository / "weights" / MODEL_KEY\n    n_estimators = int(\n        config["model_configuration"].get(\n            "n_estimators",\n            4,\n        )\n    )\n\n    feature_columns = [\n        column\n        for column in train.columns\n        if column != "target"\n    ]\n\n    start = time.perf_counter()\n\n    pipeline = TabPFNRegressorPipeline.from_pretrained(\n        device=device,\n        weights_dir=weights_dir,\n        allow_download=True,\n        n_estimators=n_estimators,\n        random_state=int(config["seed"]),\n    )\n\n    pipeline.fit(\n        train[feature_columns],\n        train["target"],\n        target_column="target",\n    )\n\n    fit_seconds = time.perf_counter() - start\n\n    start = time.perf_counter()\n    validation_prediction = pipeline.predict_values(\n        validation[feature_columns]\n    )\n    validation_prediction_seconds = (\n        time.perf_counter() - start\n    )\n\n    test_prediction = None\n    test_prediction_seconds = None\n\n    if test is not None:\n        start = time.perf_counter()\n        test_prediction = pipeline.predict_values(\n            test[feature_columns]\n        )\n        test_prediction_seconds = (\n            time.perf_counter() - start\n        )\n\n    result = {\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n        "model_licence": MODEL_LICENSE,\n        "effective_mode": "in_context",\n        "device": device,\n        "runtime_seconds": {\n            "fit": fit_seconds,\n            "validation_prediction": validation_prediction_seconds,\n            "test_prediction": test_prediction_seconds,\n        },\n        "software_versions": {\n            "python": platform.python_version(),\n            "tabpfn": package_version("tabpfn"),\n            "torch": package_version("torch"),\n            "pandas": package_version("pandas"),\n        },\n    }\n\n    pipeline.save_artifact(output_dir / "artifact" / "tabpfn")\n    artifact_metadata(output_dir,"tabpfn_native",feature_columns,device)\n    result["training_audit"]={"train_rows_supplied":len(train),"registered_support_rows":len(train)}\n    return result, validation_prediction, test_prediction\n\ndef run_tabicl(\n    config: dict[str, Any],\n    train: pd.DataFrame,\n    validation: pd.DataFrame,\n    test: pd.DataFrame | None,\n    output_dir: Path,\n    device: str,\n) -> tuple[dict[str, Any], np.ndarray, np.ndarray | None]:\n    from tabicl_regressor_pipeline.api import (\n        MIN_EVAL_ROWS,\n        MODEL_ID,\n        MODEL_KEY,\n        MODEL_LICENSE,\n        MODEL_REVISION,\n        TabICLRegressionPipeline,\n        apply_categorical_encoder,\n        create_finetuned_regressor,\n        create_regressor,\n        fine_tune_regressor,\n        fit_categorical_encoder,\n        prepare_regression_table,\n    )\n\n    repository = Path(config["repository_path"])\n    weights_dir = repository / "weights" / MODEL_KEY\n    settings = config["model_configuration"]\n    n_estimators = int(\n        settings.get("n_estimators", 8)\n    )\n\n    train_clean, _ = prepare_regression_table(\n        train,\n        "target",\n    )\n    validation_clean, _ = prepare_regression_table(\n        validation,\n        "target",\n        min_rows=MIN_EVAL_ROWS,\n    )\n\n    test_clean = None\n\n    if test is not None:\n        test_clean, _ = prepare_regression_table(\n            test,\n            "target",\n            min_rows=MIN_EVAL_ROWS,\n        )\n\n    feature_columns = [\n        column\n        for column in train_clean.columns\n        if column != "target"\n    ]\n\n    encoders = fit_categorical_encoder(\n        train_clean,\n        feature_columns,\n    )\n\n    X_train, _ = apply_categorical_encoder(\n        train_clean[feature_columns],\n        encoders,\n    )\n    X_validation, _ = apply_categorical_encoder(\n        validation_clean[feature_columns],\n        encoders,\n    )\n    X_test = None\n\n    if test_clean is not None:\n        X_test, _ = apply_categorical_encoder(\n            test_clean[feature_columns],\n            encoders,\n        )\n\n    start = time.perf_counter()\n\n    base_pipeline = TabICLRegressionPipeline.from_pretrained(\n        weights_dir=weights_dir,\n        allow_download=True,\n        n_estimators=n_estimators,\n        random_state=int(config["seed"]),\n        device=device,\n    )\n\n    requested_mode = config["condition"]\n\n    if requested_mode == "fine_tuned":\n        if device != "cuda":\n            raise RuntimeError(\n                "TabICLv2 fine-tuning requires CUDA. "\n                "Run the in-context condition on CPU or switch to a GPU runtime."\n            )\n\n        fine_tune_directory = (\n            output_dir / "model_work" / "finetune"\n        )\n        fine_tune_directory.mkdir(\n            parents=True,\n            exist_ok=True,\n        )\n\n        finetuner = create_finetuned_regressor(\n            epochs=int(\n                settings.get("fine_tune_epochs", 10)\n            ),\n            learning_rate=1e-5,\n            weight_decay=0.01,\n            n_estimators_finetune=1,\n            n_estimators_validation=1,\n            n_estimators_inference=n_estimators,\n            early_stopping=True,\n            patience=int(\n                settings.get("fine_tune_patience", 3)\n            ),\n            time_limit=int(\n                settings.get(\n                    "fine_tune_time_limit",\n                    600,\n                )\n            ),\n            eval_metric="mae",\n            model_path=str(base_pipeline.model_path),\n            allow_auto_download=False,\n            device="cuda",\n            random_state=int(config["seed"]),\n            verbose=True,\n        )\n\n        fine_tune_regressor(\n            finetuner,\n            X_train,\n            train_clean["target"],\n            X_val=X_validation,\n            y_val=validation_clean["target"],\n            output_dir=str(fine_tune_directory),\n        )\n\n        candidate_checkpoint = (\n            fine_tune_directory / "best.ckpt"\n        )\n\n        if not candidate_checkpoint.exists():\n            raise RuntimeError(\n                "TabICLv2 fine-tuning did not produce best.ckpt."\n            )\n\n        active_pipeline = TabICLRegressionPipeline(\n            create_regressor(\n                model_path=candidate_checkpoint,\n                allow_auto_download=False,\n                n_estimators=n_estimators,\n                random_state=int(config["seed"]),\n                device=device,\n            ),\n            model_path=candidate_checkpoint,\n            n_estimators=n_estimators,\n            random_state=int(config["seed"]),\n            device=device,\n            source="fine-tuned",\n        )\n        active_pipeline.fit(\n            X_train,\n            train_clean["target"],\n        )\n        effective_mode = "fine_tuned"\n    else:\n        active_pipeline = base_pipeline.fit(\n            X_train,\n            train_clean["target"],\n        )\n        effective_mode = "in_context"\n\n    fit_seconds = time.perf_counter() - start\n\n    start = time.perf_counter()\n    validation_prediction = active_pipeline.predict(\n        X_validation\n    )\n    validation_prediction_seconds = (\n        time.perf_counter() - start\n    )\n\n    test_prediction = None\n    test_prediction_seconds = None\n\n    if X_test is not None:\n        start = time.perf_counter()\n        test_prediction = active_pipeline.predict(\n            X_test\n        )\n        test_prediction_seconds = (\n            time.perf_counter() - start\n        )\n\n    result = {\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n        "model_licence": MODEL_LICENSE,\n        "effective_mode": effective_mode,\n        "device": device,\n        "runtime_seconds": {\n            "fit": fit_seconds,\n            "validation_prediction": validation_prediction_seconds,\n            "test_prediction": test_prediction_seconds,\n        },\n        "software_versions": {\n            "python": platform.python_version(),\n            "tabicl": package_version("tabicl"),\n            "torch": package_version("torch"),\n            "pandas": package_version("pandas"),\n        },\n    }\n\n    save_pickle_artifact(output_dir,{"model":active_pipeline,"encoders":encoders},\n                         feature_columns,device)\n    result["training_audit"]={"train_rows_supplied":len(train),"registered_support_rows":len(X_train),\n       "internal_validation_policy":"explicit_val_csv" if requested_mode=="fine_tuned" else "no_gradient_updates"}\n    return result, validation_prediction, test_prediction\n\n\ndef execute_config(config):\n    out=Path(config["output_dir"]); out.mkdir(parents=True,exist_ok=True)\n    paths={k:Path(v) for k,v in config["split_paths"].items()}\n    for split,path in paths.items():\n        if sha256_file(path) != config["identity"]["split_sha256"][split]:\n            raise ValueError(f"Changed {split} CSV; experiment inputs are no longer the recorded bytes.")\n    device=resolve_device(config["device_preference"])\n    seed_all(config["seed"])\n    features=config["identity"]["features"]\n    validation=pd.read_csv(paths["val"],float_precision="round_trip")\n    phase=config["phase"]\n    if phase == "development":\n        train=pd.read_csv(paths["train"],float_precision="round_trip")\n        train=train[features+["target"]]\n        validation_model=validation[features+["target"]]\n        runner={"mitra":run_mitra,"tabdpt":run_tabdpt,"tabpfn":run_tabpfn,"tabicl":run_tabicl}[config["model_family"]]\n        result,pred,_=runner(config,train,validation_model,None,out,device)\n        pd_out=prediction_frame(validation,pred)\n        pd_out.to_csv(out / "validation_predictions.csv",index=False)\n        # Verify the serialized object is usable before registering a successful development run.\n        restored=load_predictor(out,device)\n        seed_all(config["seed"])\n        start=time.perf_counter()\n        roundtrip=np.asarray(restored(validation[features]),dtype=float).reshape(-1)\n        np.testing.assert_allclose(roundtrip,pred,rtol=1e-5,atol=1e-7,\n                                   err_msg="Saved artifact failed validation prediction roundtrip.")\n        result["reload_validation_check"]={"passed":True,"rows":len(validation),"rtol":1e-5,"atol":1e-7}\n        result["runtime_seconds"]["artifact_roundtrip"]=time.perf_counter()-start\n        result["partition_metrics"]={"validation":regression_metrics(validation.target,pred),"test":None}\n    elif phase == "test":\n        # Only receipt paths supplied by the already-validated freeze reach this branch.\n        receipt_path=Path(config["source_receipt"])\n        receipt=validate_receipt(receipt_path,expected_receipt_sha=config["source_receipt_sha256"],\n                                 expected_fingerprint=config["source_fingerprint"])\n        root=receipt_path.parent\n        original=json.loads((root / "result.json").read_text())\n        if original["device"] != device: raise ValueError("Runtime device differs from frozen model.")\n        for package,version in original["software_versions"].items():\n            observed=platform.python_version() if package=="python" else package_version(package)\n            if observed != version: raise ValueError(f"Frozen runtime package changed: {package}")\n        predict=load_predictor(root,device)\n        original_predictions=canonical_predictions(pd.read_csv(root / "validation_predictions.csv"),validation)\n        seed_all(config["seed"])\n        reloaded=np.asarray(predict(validation[features]),dtype=float).reshape(-1)\n        np.testing.assert_allclose(reloaded,original_predictions.prediction,rtol=1e-5,atol=1e-7)\n        test=pd.read_csv(paths["test"],float_precision="round_trip")\n        seed_all(config["seed"])\n        start=time.perf_counter(); pred=predict(test[features]); seconds=time.perf_counter()-start\n        prediction_frame(test,pred).to_csv(out / "test_predictions.csv",index=False)\n        result={k:original[k] for k in ("model_id","model_revision","model_licence","effective_mode","device","training_audit")}\n        result["partition_metrics"]={"validation":None,"test":regression_metrics(test.target,pred)}\n        result["runtime_seconds"]={"fit":0.0,"test_prediction":seconds}\n        result["evaluation_protocol"]="loaded_selected_artifact_no_refit"\n        result["source_receipt_sha256"]=config["source_receipt_sha256"]\n        result["freeze_sha256"]=config["freeze_sha256"]\n    else: raise ValueError("Unknown phase")\n    result.update(model_key=config["model_key"],model=config["display_name"],condition=config["condition"],\n                  family="Tabular foundation model",repository=config["repository"],repository_commit=config["repository_commit"],\n                  identity_fingerprint=config["identity_fingerprint"],seed=config["seed"],\n                  software_versions={"python":platform.python_version(), **{x:package_version(x) for x in\n                                    ("torch","pandas","numpy","scikit-learn","autogluon.tabular","tabdpt","tabpfn","tabicl")}})\n    write_json(out / "result.json",result)\n    print(json.dumps({"model":result["model"],"mode":result["effective_mode"],"phase":phase,\n                       "metrics":result["partition_metrics"]},indent=2))\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument("--config",required=True);args=p.parse_args()\n    execute_config(json.loads(Path(args.config).read_text()))\n\nif __name__=="__main__": main()\n'
RUNNER_SCRIPT.write_bytes(RUNNER_SOURCE.encode("utf-8"))
compile(RUNNER_SOURCE,str(RUNNER_SCRIPT),"exec")
RUNNER_SHA256=sha256_file(RUNNER_SCRIPT)
print("Model runner ready; source SHA-256:",RUNNER_SHA256)

## 5.4 Run models on training and validation data
This stage never scores test targets. Results are reused only when the dataset and split hashes, features, parameters, seed, model/code identity, and environment signature match and all recorded artifacts are intact.

Changing a model parameter creates a new run. A changed dataset or seed requires a new experiment in Section 0.3. A frozen experiment rejects further fitting. Checkpoint downloads and first-load overhead can make the first run slower than a cached run.

In [ ]:
# @title 5.4 Execute validation-stage foundation-model runs locally
assert_current_data();SESSION.assert_development()

def foundation_controls(key, features=None):
    spec=CONDITION_SPECS[key];env=spec["environment"]
    if env not in ENVIRONMENT_RECORDS:
        raise ValueError(f"Prepare the {env} environment in 5.2 first.")
    if sha256_file(RUNNER_SCRIPT)!=RUNNER_SHA256:
        raise ValueError("Runner source changed; start a new experiment.")
    return {**base_controls(features),"backend":"foundation","environment":env,
            "parameters":copy.deepcopy(spec["configuration"]),"condition":spec["condition"],
            "repository":MODEL_REPOSITORIES[env]["repository"],"repository_commit":MODEL_REPOSITORIES[env]["commit"],
            "checkpoint":MODEL_PROVENANCE[env],"environment_signature":ENVIRONMENT_RECORDS[env]["signature"],
            "device_preference":DEVICE_PREFERENCE,"runner_sha256":RUNNER_SHA256}

ENGINE=LocalExperiment(SESSION,dataset_paths,RUNNER_SCRIPT,SUPPORT_DIR / "workshop_core.py",
                      resolve_environment=describe_environment,execute_command=stream_command)

FOUNDATION_RECEIPTS=[]
FOUNDATION_RUN_ERRORS={}
successful_foundation_keys=[]

for key in selected_foundation_conditions:
    print(f"\n=== Running {key} ===")
    try:
        receipt=ENGINE.run_validation(
            key,
            foundation_controls(key),
            CONDITION_SPECS[key],
            force=FORCE_MODEL_RERUN,
        )
        FOUNDATION_RECEIPTS.append(receipt)
        successful_foundation_keys.append(key)
    except Exception as err:
        FOUNDATION_RUN_ERRORS[key]=f"{type(err).__name__}: {err}"
        print(f"ERROR running {key}: {FOUNDATION_RUN_ERRORS[key]}")

CURRENT_FOUNDATION_KEYS=list(successful_foundation_keys)

if FOUNDATION_RUN_ERRORS:
    print("\nFoundation-model runtime failures:")
    display(pd.Series(FOUNDATION_RUN_ERRORS,name="error").to_frame())

print(
    f"Successful current foundation-model runs: "
    f"{len(FOUNDATION_RECEIPTS)} / {len(selected_foundation_conditions)}"
)

if selected_foundation_conditions and not FOUNDATION_RECEIPTS:
    raise RuntimeError(
        "All selected foundation-model runs failed. "
        "The notebook will not present a baseline-only table as a multi-model comparison. "
        "Review the errors above and the per-model install/run logs."
    )

if len(selected_foundation_conditions) >= 2 and len(FOUNDATION_RECEIPTS) < 2:
    print(
        "WARNING: fewer than two foundation-model conditions succeeded. "
        "This does not satisfy the workshop's multi-model comparison objective."
    )


In [ ]:
# @title 5.5 Load and validate generated prediction files
foundation_validation_results,prediction_registry=load_results(FOUNDATION_RECEIPTS,frames["val"])
print("Prediction schema: row_id, target, prediction. Stockout features are joined from the reference split.")
display(foundation_validation_results[["model","condition","mae","rmse","median_absolute_error","r2","device","run_id"]].round(5))

In [ ]:
# @title 5.6 Compare classical and foundation-model validation results
combined_validation_results=normalize_results(pd.concat([baseline_validation_results,foundation_validation_results],ignore_index=True))
combined_validation_results=combined_validation_results.sort_values(PRIMARY_METRIC,ascending=PRIMARY_METRIC!="r2")
display(combined_validation_results[["model","condition","mae","rmse","median_absolute_error","r2","runtime_seconds","device"]].round(5))
plt.figure(figsize=(10,6))
plt.barh(combined_validation_results.model+" / "+combined_validation_results.condition,combined_validation_results.mae)
plt.title("Validation MAE: classical and foundation-model conditions")
plt.xlabel("MAE (normalized observed-sales scale)");plt.tight_layout();show_and_save_plot(run_ids=combined_validation_results.run_id.dropna().tolist())
print("Runtime is descriptive: first-run fit timing may include checkpoint loading/download. It is not an equal-hardware speed benchmark.")

## Multi-model checkpoint

Use the comparison table to answer:

1. Which condition has the lowest validation MAE?
2. Does that condition also have the lowest RMSE?
3. Does a foundation model outperform LightGBM and Random Forest?
4. Did a fine-tuned condition actually update weights?
5. How much additional runtime was required for any improvement?
6. Is the improvement large enough to matter for the purpose of this training exercise?
7. Which licence restrictions must be retained in the report?

Avoid writing “Model X is the best tabular model.” The evidence supports only a statement about this dataset version, split, metric, repository revision, and run configuration.

Use the observed row-count audit in each `result.json`. An in-context model may use fewer rows per prediction than the number registered for the model; do not assume identical effective context sizes.

# 6. Optional extension: stockout-feature ablation

An **ablation experiment** removes one feature group while holding the rest of the procedure constant. It helps test whether that feature group contributes useful predictive information.

Here, remove:

- `stockout_hours`; and
- `roll_7_stockout`.

The comparison does not prove a causal effect of stockouts. It tests whether those variables improve prediction under this fixed modeling protocol.

In [ ]:
# @title 6.1 Optional classical stockout-feature ablation
RUN_LOCAL_ABLATION = True # @param {type:"boolean"}
ABLATION_MODEL = "random_forest" # @param ["ridge", "random_forest", "lightgbm"]
STOCKOUT_FEATURES=["stockout_hours","roll_7_stockout"]
ablation_results=normalize_results([])
CURRENT_ABLATION_KEYS=[];ABLATION_RECEIPTS=[]
if RUN_LOCAL_ABLATION:
    assert_current_data();SESSION.assert_development()
    if ABLATION_MODEL not in model_templates: raise ValueError("Choose a baseline that ran successfully.")
    reduced=[f for f in feature_columns if f not in STOCKOUT_FEATURES]
    model=clone(model_templates[ABLATION_MODEL]);start=time.perf_counter();model.fit(X_train[reduced],y_train)
    elapsed=time.perf_counter()-start
    key=ABLATION_MODEL+"_no_stockout";ctrl=classical_controls(key,reduced)
    identity={"experiment_id":SESSION.experiment_id,"model_key":key,"controls":ctrl,
              "split_sha256":DATASET_IDENTITY["split_sha256"],"features":reduced}
    receipt=save_classical_run(SESSION,key,model,frames["train"],frames["val"],ctrl,fit_seconds=elapsed,
                             identity=identity,model_name=names[ABLATION_MODEL]+" without stockout features")
    ABLATION_RECEIPTS.append(receipt);CURRENT_ABLATION_KEYS.append(key)
    tested,_=load_results([receipt],frames["val"])
    ablation_results=normalize_results(pd.concat([baseline_validation_results[baseline_validation_results.model_key.eq(ABLATION_MODEL)],tested]))
    display(ablation_results[["model","mae","rmse","r2"]].round(5))
else:print("Classical ablation skipped.")

### Foundation-model stockout ablation

The next cell repeats one selected foundation-model condition after removing:

```text
stockout_hours
roll_7_stockout
```

Every other model setting remains unchanged. The environment and weights are reused, but the model is conditioned or fitted again on the reduced feature table.

This is a predictive ablation. It tests whether those columns improve this model's validation performance under the fixed protocol. It does not establish that stockouts causally change demand.

In [ ]:
# @title 6.2 Optional foundation-model stockout-feature ablation
RUN_FOUNDATION_ABLATION = True # @param {type:"boolean"}
FOUNDATION_ABLATION_CONDITION = "" # @param {type:"string"}
# @markdown Blank uses the first successful foundation-model condition.
foundation_ablation_results=normalize_results([])
# Re-running this cell replaces this session's active FM-ablation selection only.
CURRENT_ABLATION_KEYS=[k for k in CURRENT_ABLATION_KEYS if not k.startswith("fm_ablation_")]
if RUN_FOUNDATION_ABLATION:
    assert_current_data();SESSION.assert_development()
    candidate=choose_diagnostic_model(prediction_registry,FOUNDATION_ABLATION_CONDITION)
    if candidate is None:print("Run a foundation model first; no foundation ablation executed.")
    else:
        reduced=[f for f in feature_columns if f not in STOCKOUT_FEATURES]
        key="fm_ablation_"+candidate
        ctrl=foundation_controls(candidate,reduced)
        receipt=ENGINE.run_validation(key,ctrl,CONDITION_SPECS[candidate],force=FORCE_MODEL_RERUN)
        ABLATION_RECEIPTS.append(receipt);CURRENT_ABLATION_KEYS.append(key)
        tested,_=load_results([receipt],frames["val"])
        foundation_ablation_results=normalize_results(pd.concat([
            foundation_validation_results[foundation_validation_results.model_key.eq(candidate)],tested]))
        display(foundation_ablation_results[["model_key","mae","rmse","r2"]].round(5))
else:print("Foundation-model ablation skipped.")

# 7. Optional prediction-level error analysis

In [ ]:
# @title 7.1 Inspect a locally executed foundation model's errors

PREDICTION_MODEL_KEY = "" # @param {type:"string"}
TARGET_QUANTILE_BINS = 5 # @param {type:"slider", min:3, max:10, step:1}

PREDICTION_MODEL_KEY=choose_diagnostic_model(prediction_registry,PREDICTION_MODEL_KEY)
if PREDICTION_MODEL_KEY is None:
    print("No foundation model ran. Use the baseline diagnostics in 4.4, or run a foundation model in 5.4.")
else:
    diagnostics = prediction_registry[
        PREDICTION_MODEL_KEY
    ].copy()

    diagnostics["residual"] = (
        diagnostics["target"]
        - diagnostics["prediction"]
    )
    diagnostics["absolute_error"] = diagnostics[
        "residual"
    ].abs()

    lower = float(
        min(
            diagnostics["target"].min(),
            diagnostics["prediction"].min(),
        )
    )
    upper = float(
        max(
            diagnostics["target"].max(),
            diagnostics["prediction"].max(),
        )
    )

    plt.figure(figsize=(7, 6))
    plt.scatter(
        diagnostics["target"],
        diagnostics["prediction"],
        alpha=0.5,
    )
    plt.plot([lower, upper], [lower, upper], linestyle="--")
    plt.title(
        f"Observed versus predicted: "
        f"{PREDICTION_MODEL_KEY}"
    )
    plt.xlabel("Observed target")
    plt.ylabel("Predicted target")
    plt.tight_layout()
    show_and_save_plot(run_ids=[SESSION.current_record(PREDICTION_MODEL_KEY)[1]["run_id"]])

    plt.figure(figsize=(8, 5))
    plt.hist(diagnostics["residual"], bins=40)
    plt.axvline(0, linewidth=1)
    plt.title(
        f"Residual distribution: "
        f"{PREDICTION_MODEL_KEY}"
    )
    plt.xlabel("Residual = observed - predicted")
    plt.ylabel("Count")
    plt.tight_layout()
    show_and_save_plot(run_ids=[SESSION.current_record(PREDICTION_MODEL_KEY)[1]["run_id"]])

    diagnostics["target_band"] = pd.qcut(
        diagnostics["target"],
        q=TARGET_QUANTILE_BINS,
        duplicates="drop",
    )

    target_band_error = (
        diagnostics
        .groupby("target_band", observed=True)["absolute_error"]
        .mean()
        .reset_index()
    )

    display(target_band_error)

    plt.figure(figsize=(9, 5))
    plt.bar(
        target_band_error["target_band"].astype(str),
        target_band_error["absolute_error"],
    )
    plt.title(
        f"Mean absolute error by observed-target band: "
        f"{PREDICTION_MODEL_KEY}"
    )
    plt.xlabel("Observed-target quantile band")
    plt.ylabel("Mean absolute error")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    show_and_save_plot(run_ids=[SESSION.current_record(PREDICTION_MODEL_KEY)[1]["run_id"]])

    if "stockout_hours" in diagnostics.columns:
        diagnostics["stockout_band"] = pd.qcut(
            diagnostics["stockout_hours"],
            q=4,
            duplicates="drop",
        )

        stockout_error = (
            diagnostics
            .groupby("stockout_band", observed=True)["absolute_error"]
            .mean()
            .reset_index()
        )

        display(stockout_error)

        plt.figure(figsize=(9, 5))
        plt.bar(
            stockout_error["stockout_band"].astype(str),
            stockout_error["absolute_error"],
        )
        plt.title(
            f"Mean absolute error by stockout exposure: "
            f"{PREDICTION_MODEL_KEY}"
        )
        plt.xlabel("Stockout-hours band")
        plt.ylabel("Mean absolute error")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        show_and_save_plot(run_ids=[SESSION.current_record(PREDICTION_MODEL_KEY)[1]["run_id"]])
    else:
        print(
            "Stockout-stratified analysis is unavailable because "
            "the prediction export was not aligned to validation features."
        )

# 8. Freeze the experiment and evaluate the test set

## Freeze the experiment before seeing test results
Use validation results to choose the model conditions you will evaluate. Freezing records the exact successful local run IDs, CSV hashes, feature sets, parameters, code/environment identities, and saved model artifact hashes.

**After freezing, you cannot fit again or change settings in this experiment.** Start a new experiment in Section 0.3 to explore different choices; earlier results remain on disk. Final testing loads the saved models. It does not repeat training or fine-tuning.

The freeze is an experiment-control mechanism, not an access-control system: you already possess the ZIP, including test labels. Do not inspect those labels during model development. An EDA run that exposed test targets must be described as exploratory, not an untouched-test experiment.

Leave **Freeze now** off while exploring. When ready, select successful local model keys below and turn it on. Blank selection freezes all current successful full-feature runs; ablations must be selected explicitly by their displayed key.

In [ ]:
# @title 8.0 Freeze selected successful local runs
FREEZE_NOW = False # @param {type:"boolean"}
SELECTED_MODEL_KEYS = "" # @param {type:"string"}
# @markdown Comma-separated successful local keys, or blank for current full-feature runs.

def desired_controls_for(key):
    if key.startswith("fm_ablation_"):
        return foundation_controls(key[len("fm_ablation_"):],[f for f in feature_columns if f not in STOCKOUT_FEATURES])
    if key in CONDITION_SPECS:return foundation_controls(key)
    return classical_controls(key,[f for f in feature_columns if f not in STOCKOUT_FEATURES] if key.endswith("_no_stockout") else None)

def current_controls_for(keys):
    assert_current_data()
    return {key:desired_controls_for(key) for key in keys}

eligible_keys=list(CURRENT_CLASSICAL_KEYS)+list(CURRENT_FOUNDATION_KEYS)+list(CURRENT_ABLATION_KEYS)
print("Successful/current candidate keys:",eligible_keys)
selected_keys=[k.strip() for k in SELECTED_MODEL_KEYS.split(",") if k.strip()] or list(CURRENT_CLASSICAL_KEYS)+list(CURRENT_FOUNDATION_KEYS)
if FREEZE_NOW:
    if any(k not in eligible_keys for k in selected_keys):raise ValueError("Select only current successful model keys.")
    if INCLUDE_TEST_TARGETS_IN_EDA or TEST_TARGETS_VIEWED:
        raise ValueError("Test targets were enabled in EDA. Keep this as an exploratory exercise; start a fresh untouched-test study for final claims.")
    FROZEN_MANIFEST=SESSION.freeze(selected_keys,current_controls_for(selected_keys))
    print("Frozen manifest:",FROZEN_MANIFEST)
    print("Frozen models:",selected_keys)
else:print("Not frozen yet. Finish development before enabling Freeze now.")

In [ ]:
# @title 8.1 Evaluate only the frozen models on the test partition
EVALUATE_FROZEN_TEST = False # @param {type:"boolean"}
# @markdown Requires Section 8.0. Cached test results are reused only for this same freeze.
FINAL_RECEIPTS=[]
if EVALUATE_FROZEN_TEST:
    assert_current_data()
    if not (SESSION.root / "freeze.json").exists():raise ValueError("Freeze successful runs in 8.0 first.")
    frozen=json.loads((SESSION.root / "freeze.json").read_text())
    frozen_keys=list(frozen["selected"])
    if SELECTED_MODEL_KEYS.strip() and set(k.strip() for k in SELECTED_MODEL_KEYS.split(",") if k.strip())!=set(frozen_keys):
        raise ValueError("Selection changed after freeze. No test evaluation started.")
    controls=current_controls_for(frozen_keys)
    SESSION.validate_freeze(controls)
    for key in frozen_keys:
        if controls[key]["backend"]=="classical":
            FINAL_RECEIPTS.append(evaluate_classical_frozen(SESSION,key,controls,frames["val"],frames["test"]))
        else:FINAL_RECEIPTS.append(ENGINE.run_test(key,controls))
    TEST_EVALUATION_COMPLETED=True
    final_test_results,foundation_test_predictions=load_results(FINAL_RECEIPTS,frames["test"],partition="test")
else:
    final_test_results=normalize_results([])
    TEST_EVALUATION_COMPLETED=False
    print("Test evaluation is off. Models will not be refitted when it is enabled.")

In [ ]:
# @title 8.2 Display frozen-test results (baseline-only and partial-model selections supported)
final_test_results=normalize_results(final_test_results)
if final_test_results.empty:print("No final-test results to display. Complete the freeze and evaluation steps first.")
else:
    final_test_results=final_test_results.sort_values(PRIMARY_METRIC,ascending=PRIMARY_METRIC!="r2")
    display(final_test_results[["model","condition","mae","rmse","median_absolute_error","r2","device","run_id"]].round(5))
    plt.figure(figsize=(10,6))
    plt.barh(final_test_results.model+" / "+final_test_results.condition,final_test_results.mae)
    plt.title("Frozen-model test MAE — no refitting")
    plt.xlabel("MAE (normalized observed-sales scale)");plt.tight_layout();show_and_save_plot(run_ids=final_test_results.run_id.dropna().tolist())

# 9. Interpret the evidence

## Write an evidence-based conclusion

Use this structure:

### 1. State the question

> We tested whether the selected tabular foundation models running locally improved seven-day-ahead observed-sales prediction relative to naive and classical machine-learning baselines.

### 2. Report the primary result

> On the fixed test partition, **[model and condition]** obtained an MAE of **[value]**, compared with **[value]** for **[reference baseline]**.

### 3. Report supporting metrics

> Its RMSE was **[value]**, median absolute error was **[value]**, and \(R^2\) was **[value]**.

### 4. Describe adaptation honestly

> The run used **[in-context learning / effective fine-tuning]**, as confirmed by **[run metadata]**.

### 5. Interpret stockout findings

> Removing stockout features **[increased / decreased / did not materially change]** validation MAE by **[value]**.

### 6. State limitations

At minimum, discuss:

- the sample includes only 220 store–product series;
- it is derived from one fresh-retail source;
- it covers a short time window;
- the target is normalized observed sales;
- stockouts may censor latent demand;
- the sample is intended for testing and demonstration;
- the model comparison depends on the selected metric and run configuration; and
- foundation-model pretraining overlap may affect benchmark interpretation.

### 7. Avoid overclaiming

Prefer:

> “Under this dataset version and evaluation protocol, Model A produced lower MAE than Model B.”

Avoid:

> “Model A is the best tabular model.”

Also avoid calling the target “true demand.” It is observed future `sale_amount`, which may be constrained by product availability.

## Participant submission checklist

Submit the following:

- [ ] Dataset repository revision and SHA-256 digest
- [ ] Software versions
- [ ] One target-distribution visualization
- [ ] One feature-shift visualization
- [ ] One feature–target visualization
- [ ] Classical baseline results
- [ ] At least two foundation-model conditions
- [ ] Effective adaptation mode for each local run
- [ ] Validation comparison table
- [ ] Final test comparison table
- [ ] Stockout ablation result or a justified reason for omitting it
- [ ] One evidence-based conclusion
- [ ] At least three limitations
- [ ] Model and dataset licence notes

### Suggested scoring guide

| Criterion | Weight |
|---|---:|
| Reproducible setup and provenance | 15% |
| Correct use of train, validation, and test partitions | 20% |
| EDA quality and interpretation | 20% |
| Controlled model comparison | 20% |
| Error and stockout analysis | 10% |
| Evidence-based conclusion and limitations | 15% |


# 10. Export the experiment record

In [ ]:
# @title 10.1 Export only current registered results and generated figures
CREATE_ZIP_BUNDLE = True # @param {type:"boolean"}
DOWNLOAD_ZIP_BUNDLE = False # @param {type:"boolean"}
INCLUDE_FROZEN_TEST_RESULTS = False # @param {type:"boolean"}
assert_current_data()
current_keys=list(CURRENT_CLASSICAL_KEYS)+list(CURRENT_FOUNDATION_KEYS)+list(CURRENT_ABLATION_KEYS)
if INCLUDE_FROZEN_TEST_RESULTS:
    if not TEST_EVALUATION_COMPLETED:raise ValueError("Complete frozen-test evaluation first, or leave test export off.")
    frozen=json.loads((SESSION.root / "freeze.json").read_text())
    export_keys=list(frozen["selected"])
else:export_keys=current_keys
controls=current_controls_for(export_keys)
receipts=SESSION.export_records(export_keys,controls,include_test=INCLUDE_FROZEN_TEST_RESULTS)
export_directory=new_owned_directory(SESSION.root / "exports","report")
export_run_files(receipts,export_directory)
# Recompute tables from the accepted receipt list; never export stale in-memory model tables.
development=[p for p in receipts if validate_receipt(p)["stage"]=="development"]
accepted_validation,_=load_results(development,frames["val"])
accepted_validation.to_csv(export_directory / "validation_results.csv",index=False)
if INCLUDE_FROZEN_TEST_RESULTS:
    test_receipts=[p for p in receipts if validate_receipt(p)["stage"]=="test"]
    accepted_test,_=load_results(test_receipts,frames["test"],partition="test")
    accepted_test.to_csv(export_directory / "test_results.csv",index=False)
    shutil.copy2(SESSION.root / "freeze.json",export_directory / "freeze.json")
for name,table in (("dataset_quality",quality_report),("target_summary",target_summary if INCLUDE_FROZEN_TEST_RESULTS else target_summary.drop(index="test",errors="ignore")),("feature_shift",shift_report),
                   ("feature_target_correlations",correlation_report)):
    # Test-target EDA is disallowed for the confirmatory path; summary excludes test by default.
    table.to_csv(export_directory / (name+".csv"))
figures=[]
accepted_run_ids={validate_receipt(p)["run_id"] for p in receipts}
for key,entry in FIGURE_REGISTRY.items():
    if not set(entry.get("run_ids",[])).issubset(accepted_run_ids):
        print("Omitting a chart produced by an unselected or superseded run:",entry["title"])
        continue
    # Only current experiment figures; omit test plots unless explicitly included.
    if not INCLUDE_FROZEN_TEST_RESULTS and "test" in entry["title"].lower() and "validation" not in entry["title"].lower():continue
    path=Path(entry["path"])
    if SESSION.root not in path.resolve().parents or sha256_file(path)!=entry["sha256"]:
        raise ValueError("Figure is stale, changed, or belongs to another experiment.")
    destination=export_directory / "figures" / path.name
    destination.parent.mkdir(exist_ok=True);shutil.copy2(path,destination)
    figures.append({"file":str(destination.relative_to(export_directory)),"title":entry["title"],"sha256":entry["sha256"]})
manifest={"notebook_revision":"2.1.0","experiment_id":SESSION.experiment_id,
          "base_identity":SESSION.base_identity,"dataset_provenance":DATASET_PROVENANCE,
          "current_run_ids":[validate_receipt(p)["run_id"] for p in receipts],
          "selected_model_keys":export_keys,"includes_test_results":INCLUDE_FROZEN_TEST_RESULTS,
          "freeze_sha256":SESSION.state().get("freeze_sha256") if INCLUDE_FROZEN_TEST_RESULTS else None,
          "figures":figures,"software_versions":SOFTWARE_VERSIONS,
          "artifact_policy":"Fitted models remain local; receipt inventories record their hashes."}
write_json(export_directory / "experiment_manifest.json",manifest)
print("Exported current experiment to:",export_directory)
if CREATE_ZIP_BUNDLE:
    bundle_path=Path(shutil.make_archive(str(export_directory),"zip",root_dir=export_directory))
    print("Report ZIP:",bundle_path,"SHA-256:",sha256_file(bundle_path))
    if DOWNLOAD_ZIP_BUNDLE:
        try:
            from google.colab import files
            files.download(str(bundle_path))
        except (ImportError, Exception) as err:
            print(f"Colab file download unavailable: {err}")

# Appendix A. Instructor and reproducibility notes

The workshop dataset is the pinned repository-hosted `freshretailnet-h7.zip` sample derived from FreshRetailNet-50K. DIMER is not used to acquire the dataset and no DIMER model jobs are submitted. The notebook executes the pinned model pipelines locally.

**Exercise sequence:** acquire pinned sample → verify hash/schema → EDA → validation comparisons → optional ablation → freeze selected local runs → load those same models for final testing → export registered results and saved figures.

The model recipes and dependency pins remain at the reviewed repository commits. Foundation-model environments use isolated Python 3.12 interpreters created with `uv`, which avoids dependence on the notebook host's Python minor version. Environment or model-run failures are surfaced explicitly; a complete failure cannot silently degrade into a baseline-only “multi-model” comparison.

The earlier Mitra adapter remediation remains notebook-owned rather than an undisclosed change to the remote pipeline repository. The result identifies the overlay and observed AutoGluon training/validation counts.

Cached results have a fingerprint covering the actual split bytes, feature schema, model/checkpoint/code identity, seed, parameters, and environment signature. The final-test cell checks the frozen selection and artifact inventory before loading models. There is no fallback that retrains a missing or invalid frozen model.

Report bundles contain only explicitly accepted current records. Trained weight files remain in the local session directory; they are not automatically redistributed with the report. Output directories are newly created; participant files are never recursively removed.

**Acceptance boundary:** the sample itself is a tutorial/sanity fixture, so its metrics are not benchmark evidence. Before teaching, run the chosen core models in a clean Colab session with the pinned sample and real model checkpoints; exercise optional GPU fine-tuning separately and verify the first saved-artifact reload.


# Appendix B. Dataset and model references

## Dataset

Workshop sample artifact: `kurtvalcorza/mitra-regressor-pipeline/examples/sample-data/freshretailnet-h7.zip`  
Pinned repository revision: `78e12407044bbca6ed8edbb9754bb33bf09117ac`  
Pinned archive SHA-256: `6534230e9eb6a2e212b741c4c17d897a57338323eb011f35ba2dc3fb28d8bb7b`

The sample is derived from:

Wang, Y., Gu, J., Long, L., Li, X., Shen, L., Fu, Z., Zhou, X., & Xu, J. (2025). *FreshRetailNet-50K: A Stockout-Annotated Censored Demand Dataset for Latent Demand Recovery and Forecasting in Fresh Retail*. arXiv:2505.16319.

Source dataset: `Dingdong-Inc/FreshRetailNet-50K`  
Pinned source revision used by the dataset builder: `08c1fab7f9257bc73679d415d65d644165d351d4`  
Dataset licence: CC BY 4.0

## Pinned pipeline repositories

| Model | Repository | Commit |
|---|---|---|
| Mitra Regressor | `kurtvalcorza/mitra-regressor-pipeline` | `a2b36c21fe92d9e42e2f3310c98e9e1c7fe8d7af` |
| TabDPT v1.2 Regressor | `kurtvalcorza/tabdpt-regressor-pipeline` | `21085dd2e148041d8fa88944c21532b4bdf9a7ea` |
| TabPFN-3 Regressor | `kurtvalcorza/tabpfn-regressor-pipeline` | `45dddb070326b9994a18e689add0546d30684239` |
| TabICLv2 Regressor | `kurtvalcorza/tabicl-regressor-pipeline` | `08146a323d516fb8dc9aac7f51ad17994e37294c` |

The notebook executes these pipeline repositories locally. It does not submit model jobs to DIMER.

## Adapter contract used for this revision
- AutoGluon 1.5 `TabularPredictor.fit`: https://auto.gluon.ai/1.5/api/autogluon.tabular.TabularPredictor.fit.html
- AutoGluon 1.5 trained-predictor loading: https://auto.gluon.ai/1.5/tutorials/tabular/tabular-essentials.html

These sources support the explicit `tuning_data` policy and saved-predictor loading. The workflow does not interpret a successful import as model-quality evidence.
